# LeetCode 4100 — Minimum Cost Path With Even XOR Parity (Hard)

Notebook này giảng bài **Minimum Cost Path With Even XOR Parity** (Weekly Contest 526, câu 4, độ khó Hard) cho người mới, đi từ con số 0.

Đề bài cho một **cây** có trọng số trên cạnh và giá trị trên đỉnh. Ta phải tìm **đường đi rẻ nhất** giữa hai đỉnh khác nhau sao cho **XOR** các giá trị trên đường có **số bit 1 chẵn**.

Nghe thì có vẻ phải duyệt mọi đường đi (khoảng 5 tỷ đường khi có 100 000 đỉnh!). Nhưng với hai quan sát nhỏ, bài toán trở nên rất gọn.

---

## Sau notebook này, bạn sẽ nắm được

1. Cách biểu diễn một **cây** bằng **danh sách kề** (adjacency list) và duyệt cây bằng **DFS dùng stack** (không đệ quy).
2. Phép **XOR** (`^`), cách **đếm bit 1** (`bin(x).count("1")`, `x.bit_count()`) và khái niệm **parity** (chẵn/lẻ).
3. Mẹo quan trọng: **tính chẵn/lẻ của số bit 1 trong XOR = XOR các tính chẵn/lẻ** → mỗi đỉnh chỉ còn là một bit 0/1.
4. Lý luận "**đường rẻ nhất luôn rất ngắn**" (chỉ 2 hoặc 3 đỉnh) khi mọi chi phí cạnh đều dương.
5. Kỹ thuật **giữ 2 giá trị nhỏ nhất** trong một lần duyệt để tránh duyệt mọi cặp hàng xóm.
6. Kỹ thuật tổng quát **Tree DP** (quy hoạch động trên cây) với trạng thái chẵn/lẻ — vũ khí cho các biến thể khó hơn.
7. Cách viết `class Solution`, kiểm thử bằng `assert`, đối chiếu với brute force trên hàng trăm cây ngẫu nhiên.

---

## Các cách giải sẽ học (từ dễ đến khó)

| Cách | Ý tưởng | Thời gian |
|---|---|---|
| Cách 1 | Brute force: DFS từ mọi đỉnh, tính XOR thật | `O(n²)` |
| Cách 2 | Chỉ duyệt các đường 2–3 đỉnh (ngây thơ: mọi cặp hàng xóm) | `O(n + Σ deg²)`, tệ nhất `O(n²)` |
| Cách 3 | Đường 2–3 đỉnh + giữ 2 cạnh rẻ nhất mỗi đỉnh | `O(n)` — **lời giải nộp bài** |
| Cách 4 | Tree DP với 2 trạng thái chẵn/lẻ | `O(n)` — kỹ thuật tổng quát |

Giữa Cách 1 và Cách 2 có hai **bước đột phá** (insight) — đó mới là "linh hồn" của bài.

# 1. Đề bài

Cho một **cây** gồm `n` đỉnh, đánh số từ `0` đến `n - 1`, biểu diễn bằng mảng 2 chiều `edges`.
Mỗi phần tử `edges[i] = [u_i, v_i, c_i]` là một cạnh nối đỉnh `u_i` và `v_i` với **chi phí** (cost) `c_i`.

Cho thêm mảng số nguyên `values` độ dài `n`, trong đó `values[i]` là **giá trị** của đỉnh `i`.

Một đường đi giữa **hai đỉnh phân biệt** được gọi là **Balanced-XOR** nếu: XOR (phép `^`) các giá trị của **tất cả** các đỉnh trên đường đi, **tính cả hai đầu mút**, có **số bit 1 chẵn**.

**Chi phí** của một đường đi là **tổng chi phí các cạnh** trên đường đó.

Hãy trả về **chi phí nhỏ nhất** của một đường Balanced-XOR. Nếu không tồn tại đường nào như vậy, trả về `-1`.

Lưu ý của đề: số `0` có **0 bit 1** (0 là số chẵn, nên XOR bằng 0 là hợp lệ).

---

## Skeleton (khung hàm)

Trang đề mình có trong tay không ghi tên method, nên notebook dùng tên kiểu LeetCode thường gặp. Nếu skeleton trên LeetCode có tên khác, bạn chỉ cần **giữ nguyên thân hàm** và đổi tên cho khớp.

```python
from typing import List

class Solution:
    def minCost(self, edges: List[List[int]], values: List[int]) -> int:
        ...
```

Thứ tự tham số (`edges` trước, `values` sau) theo đúng thứ tự trong đề và trong các ví dụ.

## Ví dụ trong đề

### Ví dụ 1

```text
edges  = [[0,1,4],[0,2,6],[0,3,3],[3,4,5]]
values = [3,1,7,1,3]
Output: 7
```

Các đường Balanced-XOR (đề liệt kê):

| Đường đi | XOR các giá trị | Tổng chi phí |
|---|---|---:|
| 1 - 0 - 2 | 1 XOR 3 XOR 7 = 5 | 4 + 6 = 10 |
| 1 - 0 - 3 | 1 XOR 3 XOR 1 = 3 | 4 + 3 = 7 |
| 1 - 0 - 3 - 4 | 1 XOR 3 XOR 1 XOR 3 = 0 | 4 + 3 + 5 = 12 |
| 2 - 0 - 3 | 7 XOR 3 XOR 1 = 5 | 6 + 3 = 9 |
| 2 - 0 - 3 - 4 | 7 XOR 3 XOR 1 XOR 3 = 6 | 6 + 3 + 5 = 14 |

Chi phí nhỏ nhất là `7`, đạt được bởi đường `1 - 0 - 3`.

### Ví dụ 2

```text
edges  = [[0,1,5],[1,2,10]]
values = [1,7,3]
Output: 5
```

| Đường đi | XOR các giá trị | Tổng chi phí |
|---|---|---:|
| 0 - 1 | 1 XOR 7 = 6 | 5 |
| 0 - 1 - 2 | 1 XOR 7 XOR 3 = 5 | 5 + 10 = 15 |

Chi phí nhỏ nhất là `5` (đường `0 - 1`).

### Ví dụ 3

```text
edges  = [[0,1,5],[0,2,7],[0,3,9]]
values = [1,3,5,9]
Output: -1
```

Mọi đường đi giữa hai đỉnh phân biệt đều có XOR với **số bit 1 lẻ**, nên không có đường Balanced-XOR → trả về `-1`.

In [1]:
from typing import List

# Lưu 3 ví dụ của đề dưới dạng (edges, values, expected) để dùng lại suốt notebook
EXAMPLES = [
    ([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3], 7),
    ([[0, 1, 5], [1, 2, 10]], [1, 7, 3], 5),
    ([[0, 1, 5], [0, 2, 7], [0, 3, 9]], [1, 3, 5, 9], -1),
]

for k, (edges, values, expected) in enumerate(EXAMPLES, start=1):
    n = len(values)
    print(f"Ví dụ {k}: n = {n}, số cạnh = {len(edges)}, expected = {expected}")
    print("   edges  =", edges)
    print("   values =", values)

Ví dụ 1: n = 5, số cạnh = 4, expected = 7
   edges  = [[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]]
   values = [3, 1, 7, 1, 3]
Ví dụ 2: n = 3, số cạnh = 2, expected = 5
   edges  = [[0, 1, 5], [1, 2, 10]]
   values = [1, 7, 3]
Ví dụ 3: n = 4, số cạnh = 3, expected = -1
   edges  = [[0, 1, 5], [0, 2, 7], [0, 3, 9]]
   values = [1, 3, 5, 9]


## Ràng buộc (constraints) và ý nghĩa của chúng

| Ràng buộc | Ý nghĩa thực tế |
|---|---|
| `2 <= n <= 10^5` | Có tới 100 000 đỉnh. Số cặp đỉnh là `n(n-1)/2` ≈ **5 tỷ** → **không thể** duyệt từng cặp. Cần khoảng `O(n)` hoặc `O(n log n)`. |
| `edges.length == n - 1`, `edges` là một cây hợp lệ | Đồ thị liên thông, không có chu trình → giữa 2 đỉnh bất kỳ có **đúng một** đường đi đơn. |
| `1 <= c_i <= 10^9` | Mọi chi phí cạnh đều **dương** (≥ 1). Chi tiết này sẽ là chìa khóa của Bước đột phá 2! Tổng chi phí có thể tới ~`10^14`, vượt `int` 32-bit (Python không lo, C++/Java phải dùng `long long`/`long`). |
| `values.length == n`, `1 <= values[i] <= 10^9` | Mỗi giá trị < `2^30`, nên có tối đa 30 bit. |

**Kết luận về độ phức tạp:** mọi cách "thử từng cặp đỉnh" đều là `O(n²)` ≈ `10^10` bước → quá chậm. Ta phải tìm cách `O(n)`.

Cell dưới tính nhanh vài con số trên để bạn "cảm" được độ lớn.

In [2]:
n_max = 10**5
num_pairs = n_max * (n_max - 1) // 2
max_total_cost = (n_max - 1) * 10**9

print("Số cặp đỉnh khi n = 10^5      :", f"{num_pairs:,}")
print("Tổng chi phí lớn nhất có thể  :", f"{max_total_cost:,}")
print("Giới hạn int 32-bit (2^31 - 1):", f"{2**31 - 1:,}")
print("2^30 =", f"{2**30:,}", "> 10^9 ?", 2**30 > 10**9)

Số cặp đỉnh khi n = 10^5      : 4,999,950,000
Tổng chi phí lớn nhất có thể  : 99,999,000,000,000
Giới hạn int 32-bit (2^31 - 1): 2,147,483,647
2^30 = 1,073,741,824 > 10^9 ? True


# 2. Hiểu đề bằng tay trước khi nghĩ thuật toán

Vẽ cây của **Ví dụ 1** (số trên cạnh là chi phí, `val` là giá trị đỉnh):

```text
        (1) val=1
         |
         4
         |
        (0) val=3 ----6---- (2) val=7
         |
         3
         |
        (3) val=1
         |
         5
         |
        (4) val=3
```

Có `n = 5` đỉnh nên có `5·4/2 = 10` cặp đỉnh, tức **10 đường đi**. Ta liệt kê **tất cả**, kể cả các đường không hợp lệ:

| Đường đi | XOR | Nhị phân | Số bit 1 | Chẵn? | Chi phí |
|---|---|---:|---:|:---:|---:|
| 0 - 1 | 3 ^ 1 = 2 | 10 | 1 | không | 4 |
| 0 - 2 | 3 ^ 7 = 4 | 100 | 1 | không | 6 |
| 0 - 3 | 3 ^ 1 = 2 | 10 | 1 | không | 3 |
| 0 - 3 - 4 | 3 ^ 1 ^ 3 = 1 | 1 | 1 | không | 8 |
| 1 - 0 - 2 | 1 ^ 3 ^ 7 = 5 | 101 | 2 | **có** | 10 |
| 1 - 0 - 3 | 1 ^ 3 ^ 1 = 3 | 11 | 2 | **có** | **7** |
| 1 - 0 - 3 - 4 | 1 ^ 3 ^ 1 ^ 3 = 0 | 0 | 0 | **có** | 12 |
| 2 - 0 - 3 | 7 ^ 3 ^ 1 = 5 | 101 | 2 | **có** | 9 |
| 2 - 0 - 3 - 4 | 7 ^ 3 ^ 1 ^ 3 = 6 | 110 | 2 | **có** | 14 |
| 3 - 4 | 1 ^ 3 = 2 | 10 | 1 | không | 5 |

Có đúng 5 đường hợp lệ (khớp bảng của đề) và rẻ nhất là `1 - 0 - 3` với chi phí `7`.

Cell dưới làm lại bảng này bằng code. Các đường đi được **ghi tay** (chưa cần thuật toán tìm đường) để ta chỉ tập trung vào cách tính XOR và chi phí.

In [3]:
values = [3, 1, 7, 1, 3]
# chi phí cạnh, lưu theo cặp đỉnh (u, v) với u < v
edge_cost = {(0, 1): 4, (0, 2): 6, (0, 3): 3, (3, 4): 5}

all_paths = [
    [0, 1], [0, 2], [0, 3], [0, 3, 4], [1, 0, 2],
    [1, 0, 3], [1, 0, 3, 4], [2, 0, 3], [2, 0, 3, 4], [3, 4],
]

best = None
for path in all_paths:
    x = 0
    for node in path:
        x ^= values[node]                 # XOR dồn giá trị các đỉnh
    cost = 0
    for a, b in zip(path, path[1:]):      # các cặp đỉnh liên tiếp = các cạnh
        cost += edge_cost[(min(a, b), max(a, b))]
    ones = bin(x).count("1")
    ok = ones % 2 == 0
    if ok and (best is None or cost < best):
        best = cost
    name = " - ".join(map(str, path))
    print(f"{name:<14} XOR={x:<2} bin={bin(x)[2:]:>4} bit1={ones}  chẵn={ok!s:<5}  cost={cost}")

print("Chi phí nhỏ nhất:", best)

0 - 1          XOR=2  bin=  10 bit1=1  chẵn=False  cost=4
0 - 2          XOR=4  bin= 100 bit1=1  chẵn=False  cost=6
0 - 3          XOR=2  bin=  10 bit1=1  chẵn=False  cost=3
0 - 3 - 4      XOR=1  bin=   1 bit1=1  chẵn=False  cost=8
1 - 0 - 2      XOR=5  bin= 101 bit1=2  chẵn=True   cost=10
1 - 0 - 3      XOR=3  bin=  11 bit1=2  chẵn=True   cost=7
1 - 0 - 3 - 4  XOR=0  bin=   0 bit1=0  chẵn=True   cost=12
2 - 0 - 3      XOR=5  bin= 101 bit1=2  chẵn=True   cost=9
2 - 0 - 3 - 4  XOR=6  bin= 110 bit1=2  chẵn=True   cost=14
3 - 4          XOR=2  bin=  10 bit1=1  chẵn=False  cost=5
Chi phí nhỏ nhất: 7


## Nhận xét sau khi làm tay

1. Với `n` đỉnh có `n(n-1)/2` đường đi. Ví dụ nhỏ thì dễ, nhưng `n = 10^5` thì không thể liệt kê.
2. Để biết một đường có hợp lệ không, ta đang tính **XOR thật** rồi đếm bit 1. Câu hỏi: **có thật sự cần giá trị XOR không**, hay chỉ cần biết "chẵn hay lẻ"? (→ Bước đột phá 1)
3. Nhìn cột "Chi phí": đường hợp lệ rẻ nhất `1 - 0 - 3` chỉ có **3 đỉnh**. Các đường dài hơn (4 đỉnh) đều đắt hơn. Đây có phải trùng hợp không? (→ Bước đột phá 2)

Trước khi trả lời, ta cần trang bị một số công cụ nền tảng.

# 3. Nền tảng (Foundations)

Các công cụ dùng trong notebook:

| Mục | Công cụ | Dùng ở đâu |
|---|---|---|
| 3.1 | Cây, danh sách kề `adj` | Mọi cách giải |
| 3.2 | Phép XOR `^` | Định nghĩa Balanced-XOR |
| 3.3 | Đếm bit 1: `bin(x).count("1")`, `x.bit_count()` | Kiểm tra chẵn/lẻ |
| 3.4 | Parity: `% 2` và `& 1` | Bước đột phá 1 |
| 3.5 | DFS bằng stack + mảng/biến `parent` | Cách 1, Cách 4 |
| 3.6 | Vì sao tránh đệ quy khi `n = 10^5` | Cách 1, Cách 4 |
| 3.7 | `INF = float("inf")` và giữ **2 giá trị nhỏ nhất** | Cách 3 |
| 3.8 | Thứ tự "con trước cha" (đảo ngược thứ tự DFS) | Cách 4 |

## 3.1 Cây (tree) và danh sách kề (adjacency list)

**Cây** là đồ thị có `n` đỉnh, `n - 1` cạnh, **liên thông** (đi được từ đỉnh bất kỳ tới đỉnh bất kỳ) và **không có chu trình**.

Hệ quả quan trọng: giữa hai đỉnh bất kỳ có **đúng một** đường đi không lặp đỉnh. Vì vậy "đường đi giữa u và v" là duy nhất — ta không phải chọn lối đi, chỉ phải chọn **cặp đầu mút**.

Đề cho danh sách cạnh `edges`, nhưng để "đứng ở một đỉnh và nhìn sang hàng xóm" ta cần **danh sách kề**:

```text
adj[u] = [(hàng_xóm_1, chi_phí_1), (hàng_xóm_2, chi_phí_2), ...]
```

Cây **vô hướng**, nên mỗi cạnh `[u, v, c]` phải được thêm **hai lần**: vào `adj[u]` và vào `adj[v]`.

**Bậc** (degree) `deg(u)` = số hàng xóm của `u` = `len(adj[u])`.

In [4]:
def build_adj(n, edges):
    adj = [[] for _ in range(n)]   # n danh sách rỗng ĐỘC LẬP
    for u, v, c in edges:          # "unpack" mỗi cạnh thành 3 biến
        adj[u].append((v, c))
        adj[v].append((u, c))      # đừng quên chiều ngược lại!
    return adj

edges1, values1, _ = EXAMPLES[0]
adj1 = build_adj(len(values1), edges1)
for u in range(len(values1)):
    print(f"adj[{u}] = {adj1[u]}   deg = {len(adj1[u])}")

adj[0] = [(1, 4), (2, 6), (3, 3)]   deg = 3
adj[1] = [(0, 4)]   deg = 1
adj[2] = [(0, 6)]   deg = 1
adj[3] = [(0, 3), (4, 5)]   deg = 2
adj[4] = [(3, 5)]   deg = 1


## 3.2 Phép XOR `^`

XOR so sánh **từng bit** của hai số: bit kết quả là `1` nếu hai bit **khác nhau**, là `0` nếu **giống nhau**.

| bit a | bit b | a XOR b |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Ví dụ `3 ^ 7`:

```text
  3 = 011
  7 = 111
------------
3^7 = 100 = 4
```

Bốn tính chất ta sẽ dùng:

- `a ^ 0 = a`
- `a ^ a = 0` (một số XOR với chính nó thì "triệt tiêu")
- Giao hoán: `a ^ b = b ^ a`
- Kết hợp: `(a ^ b) ^ c = a ^ (b ^ c)`

Nhờ giao hoán + kết hợp, XOR các đỉnh trên đường đi **không phụ thuộc thứ tự**: đi `1 → 0 → 3` hay `3 → 0 → 1` đều ra cùng kết quả.

In [5]:
a, b = 3, 7
print(f"{a} = {a:03b}")
print(f"{b} = {b:03b}")
print(f"{a} ^ {b} = {a ^ b} = {a ^ b:03b}")

print("5 ^ 0 =", 5 ^ 0)            # giữ nguyên
print("5 ^ 5 =", 5 ^ 5)            # triệt tiêu
print("1^3^1 =", 1 ^ 3 ^ 1, "| 3^1^1 =", 3 ^ 1 ^ 1)   # thứ tự không quan trọng

# XOR dồn cả một danh sách
x = 0
for v in [1, 3, 1, 3]:
    x ^= v
print("XOR của [1, 3, 1, 3] =", x)

3 = 011
7 = 111
3 ^ 7 = 4 = 100
5 ^ 0 = 5
5 ^ 5 = 0
1^3^1 = 3 | 3^1^1 = 3
XOR của [1, 3, 1, 3] = 0


## 3.3 Đếm số bit 1 (popcount)

"Số bit 1" của `x` (tiếng Anh: **popcount**, **set bits**) là số chữ số `1` trong biểu diễn nhị phân.

Hai cách trong Python:

1. `bin(x)` trả về **chuỗi** dạng `"0b101"`; đếm ký tự `"1"` bằng `.count("1")`. Tiền tố `"0b"` không chứa ký tự `1` nên không ảnh hưởng.
2. `x.bit_count()` — có từ **Python 3.10**, nhanh hơn (LeetCode dùng Python 3.11+ nên dùng được).

Chú ý cú pháp: viết `5.bit_count()` sẽ lỗi vì Python tưởng `5.` là số thực; phải viết `(5).bit_count()` hoặc dùng biến `x.bit_count()`.

In [6]:
for x in [0, 1, 2, 3, 5, 6, 7, 13]:
    print(f"x={x:>2}  bin(x)={bin(x):<7} bin(x).count('1')={bin(x).count('1')}  x.bit_count()={x.bit_count()}")

x= 0  bin(x)=0b0     bin(x).count('1')=0  x.bit_count()=0
x= 1  bin(x)=0b1     bin(x).count('1')=1  x.bit_count()=1
x= 2  bin(x)=0b10    bin(x).count('1')=1  x.bit_count()=1
x= 3  bin(x)=0b11    bin(x).count('1')=2  x.bit_count()=2
x= 5  bin(x)=0b101   bin(x).count('1')=2  x.bit_count()=2
x= 6  bin(x)=0b110   bin(x).count('1')=2  x.bit_count()=2
x= 7  bin(x)=0b111   bin(x).count('1')=3  x.bit_count()=3
x=13  bin(x)=0b1101  bin(x).count('1')=3  x.bit_count()=3


## 3.4 Tính chẵn/lẻ (parity): `% 2` và `& 1`

**Parity** của một số nguyên `k` là `k % 2`: bằng `0` nếu `k` chẵn, `1` nếu `k` lẻ.

Một cách khác: `k & 1` — phép **AND bit** với `1` giữ lại đúng **bit cuối cùng** của `k`, mà bit cuối chính là chẵn/lẻ. Với số không âm, `k % 2 == k & 1`.

Trong bài này, "số bit 1 chẵn" nghĩa là:

```python
x.bit_count() % 2 == 0      # hoặc   (x.bit_count() & 1) == 0
```

Ta đặt tên: **parity của một số `x`** = `x.bit_count() & 1` (0 = có số bit 1 chẵn, 1 = lẻ).

In [7]:
for k in [0, 1, 2, 3, 10, 11]:
    print(f"k={k:>2}  k % 2 = {k % 2}   k & 1 = {k & 1}   bin(k)={bin(k)}")

def parity(x):
    """0 nếu x có số bit 1 chẵn, 1 nếu lẻ."""
    return x.bit_count() & 1

print("parity của values ví dụ 1:", [parity(v) for v in [3, 1, 7, 1, 3]])

k= 0  k % 2 = 0   k & 1 = 0   bin(k)=0b0
k= 1  k % 2 = 1   k & 1 = 1   bin(k)=0b1
k= 2  k % 2 = 0   k & 1 = 0   bin(k)=0b10
k= 3  k % 2 = 1   k & 1 = 1   bin(k)=0b11
k=10  k % 2 = 0   k & 1 = 0   bin(k)=0b1010
k=11  k % 2 = 1   k & 1 = 1   bin(k)=0b1011
parity của values ví dụ 1: [0, 1, 1, 1, 0]


## 3.5 DFS (duyệt theo chiều sâu) bằng stack trên cây

**DFS** (Depth-First Search) đi sâu nhất có thể theo một nhánh, rồi quay lui. Ta cài bằng một **stack** (list Python: `append` để đẩy vào, `pop()` để lấy phần tử cuối ra).

Trên **cây**, ta không cần tập `visited` như đồ thị tổng quát: chỉ cần nhớ **đỉnh cha** (`parent`) — đỉnh ta vừa đi từ đó tới — và không quay lại nó. Vì cây không có chu trình, đây là cách duy nhất để "đi vòng lại".

Mỗi phần tử trong stack có thể mang thêm **trạng thái của đường đi** từ đỉnh xuất phát tới đỉnh hiện tại: XOR dồn và chi phí dồn. Khi đi qua cạnh `(u → v, c)`:

```text
xor_mới  = xor_cũ ^ values[v]
cost_mới = cost_cũ + c
```

Cell dưới chạy DFS từ đỉnh `1` của Ví dụ 1 — chính là cách tìm mọi đường đi bắt đầu từ `1`.

In [8]:
def dfs_from(adj, values, s):
    stack = [(s, -1, values[s], 0)]        # (đỉnh, cha, xor dồn, cost dồn)
    while stack:
        u, par, x, cost = stack.pop()
        print(f"thăm {u}: cha={par:>2}, xor từ {s} tới {u} = {x} (bit1={x.bit_count()}), cost = {cost}")
        for v, c in adj[u]:
            if v != par:                    # không quay lại cha
                stack.append((v, u, x ^ values[v], cost + c))

dfs_from(adj1, values1, 1)

thăm 1: cha=-1, xor từ 1 tới 1 = 1 (bit1=1), cost = 0
thăm 0: cha= 1, xor từ 1 tới 0 = 2 (bit1=1), cost = 4
thăm 3: cha= 0, xor từ 1 tới 3 = 3 (bit1=2), cost = 7
thăm 4: cha= 3, xor từ 1 tới 4 = 0 (bit1=0), cost = 12
thăm 2: cha= 0, xor từ 1 tới 2 = 5 (bit1=2), cost = 10


Đọc output: từ đỉnh `1` ta tới `0` (XOR 2, lẻ), `3` (XOR 3, chẵn, cost 7), `4` (XOR 0, chẵn, cost 12), `2` (XOR 5, chẵn, cost 10). Đúng các dòng bắt đầu bằng `1` trong bảng làm tay.

## 3.6 Vì sao tránh đệ quy khi `n = 10^5`?

Có thể viết DFS bằng **đệ quy** (hàm tự gọi chính nó) cho gọn. Nhưng Python giới hạn độ sâu đệ quy (mặc định của Python là `1000` tầng). Nếu cây là một **đường thẳng** `0 - 1 - 2 - ... - 99999`, DFS đệ quy sẽ sâu 100 000 tầng → `RecursionError`.

Có thể tăng giới hạn bằng `sys.setrecursionlimit`, nhưng vẫn có nguy cơ tràn bộ nhớ stack của C. **Cách an toàn: dùng stack tự quản lý** như ở 3.5. Cell dưới minh họa (bắt lỗi bằng `try/except` để notebook không dừng).

In [9]:
import sys
print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

n_line = 20000
line_adj = build_adj(n_line, [[i, i + 1, 1] for i in range(n_line - 1)])

def depth_recursive(u, par):
    best = 0
    for v, _ in line_adj[u]:
        if v != par:
            best = max(best, 1 + depth_recursive(v, u))
    return best

try:
    print("Đệ quy:", depth_recursive(0, -1))
except RecursionError as e:
    print("Đệ quy thất bại:", type(e).__name__)

def depth_iterative(adj, root):
    best = 0
    stack = [(root, -1, 0)]
    while stack:
        u, par, d = stack.pop()
        best = max(best, d)
        for v, _ in adj[u]:
            if v != par:
                stack.append((v, u, d + 1))
    return best

print("Stack tự quản lý:", depth_iterative(line_adj, 0))

Giới hạn đệ quy hiện tại: 1000
Đệ quy thất bại: RecursionError
Stack tự quản lý: 19999


## 3.7 `INF` ("chưa có") và giữ 2 giá trị nhỏ nhất trong một lần duyệt

Khi tìm **giá trị nhỏ nhất**, ta khởi tạo bằng `INF = float("inf")` — một số lớn hơn mọi số nguyên. Nếu sau khi duyệt xong vẫn còn `INF`, nghĩa là "không tìm thấy" → trả `-1`.

Ở Cách 3 ta cần, với mỗi đỉnh, **hai** chi phí nhỏ nhất `best1 <= best2` trong một luồng số đến lần lượt. Quy tắc khi một số mới `c` tới:

```text
nếu c < best1:   best2 = best1 ; best1 = c     (c thành nhỏ nhất, nhỏ nhất cũ bị đẩy xuống hạng 2)
elif c < best2:  best2 = c                     (c chỉ chen vào hạng 2)
else:            không làm gì
```

Chỉ tốn `O(1)` cho mỗi số — không cần sắp xếp cả danh sách.

In [10]:
INF = float("inf")
print("INF > 10**18 ?", INF > 10**18, "| min(INF, 7) =", min(INF, 7))

best1, best2 = INF, INF
for c in [8, 2, 9, 3, 5]:
    if c < best1:
        best2, best1 = best1, c
        action = "đẩy best1 cũ xuống best2"
    elif c < best2:
        best2 = c
        action = "chen vào best2"
    else:
        action = "bỏ qua"
    print(f"c = {c}: {action:<26} -> best1 = {best1}, best2 = {best2}")
print("Tổng 2 số nhỏ nhất:", best1 + best2, "| kiểm tra bằng sorted:", sum(sorted([8, 2, 9, 3, 5])[:2]))

INF > 10**18 ? True | min(INF, 7) = 7
c = 8: đẩy best1 cũ xuống best2   -> best1 = 8, best2 = inf
c = 2: đẩy best1 cũ xuống best2   -> best1 = 2, best2 = 8
c = 9: bỏ qua                     -> best1 = 2, best2 = 8
c = 3: chen vào best2             -> best1 = 2, best2 = 3
c = 5: bỏ qua                     -> best1 = 2, best2 = 3
Tổng 2 số nhỏ nhất: 5 | kiểm tra bằng sorted: 5


## 3.8 Thứ tự "con trước cha" (đảo ngược thứ tự DFS)

Ở Cách 4 (Tree DP), ta **treo cây** tại đỉnh `0` (gọi là **gốc**, root). Mỗi đỉnh khác có đúng một **cha** (`parent`). Thông tin của một đỉnh được tính từ thông tin của **các con**, nên phải xử lý **con trước, cha sau**.

Mẹo không cần đệ quy:

1. Chạy DFS bằng stack từ gốc, ghi lại thứ tự lấy ra khỏi stack vào list `order`. Một đỉnh luôn được lấy ra **trước** mọi con cháu của nó (vì con chỉ được đẩy vào stack sau khi cha đã được lấy ra).
2. Duyệt `reversed(order)` → mọi con cháu xuất hiện **trước** đỉnh đó. Đúng thứ tự "con trước cha".

In [11]:
def root_tree(adj, root=0):
    n = len(adj)
    parent = [-1] * n
    parent_cost = [0] * n            # chi phí cạnh nối đỉnh với cha
    order = []
    stack = [root]
    seen = [False] * n
    seen[root] = True
    while stack:
        u = stack.pop()
        order.append(u)
        for v, c in adj[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                parent_cost[v] = c
                stack.append(v)
    return parent, parent_cost, order

parent1, pcost1, order1 = root_tree(adj1, 0)
print("order          =", order1)
print("reversed(order)=", list(reversed(order1)))
print("parent         =", parent1)
print("parent_cost    =", pcost1)

order          = [0, 3, 4, 2, 1]
reversed(order)= [1, 2, 4, 3, 0]
parent         = [-1, 0, 0, 0, 3]
parent_cost    = [0, 4, 6, 3, 5]


# 4. Cách 1 — Brute force: thử mọi đường đi

## Ý tưởng

Làm đúng như định nghĩa của đề:

1. Với **mỗi** đỉnh xuất phát `s`, chạy DFS (mục 3.5) mang theo XOR dồn `x` và chi phí dồn `cost`.
2. Tại mỗi đỉnh `t != s` thăm được, đường `s → t` có XOR là `x`. Nếu `x` có số bit 1 chẵn, cập nhật đáp án bằng `cost`.
3. Cuối cùng, nếu chưa cập nhật lần nào thì trả `-1`.

Mỗi đường được xét **hai lần** (`s → t` và `t → s`), không sao: kết quả `min` không đổi.

Vì làm đúng định nghĩa, cách này **hiển nhiên đúng** — ta sẽ dùng nó làm "trọng tài" để kiểm tra các cách nhanh hơn.

In [12]:
def min_cost_bruteforce(edges: List[List[int]], values: List[int]) -> int:
    n = len(values)
    adj = build_adj(n, edges)
    best = -1
    for s in range(n):
        stack = [(s, -1, values[s], 0)]
        while stack:
            u, par, x, cost = stack.pop()
            if u != s and bin(x).count("1") % 2 == 0:   # hai đỉnh phân biệt + số bit 1 chẵn
                if best == -1 or cost < best:
                    best = cost
            for v, c in adj[u]:
                if v != par:
                    stack.append((v, u, x ^ values[v], cost + c))
    return best

for edges, values, expected in EXAMPLES:
    got = min_cost_bruteforce(edges, values)
    print(f"brute force = {got:>2} | expected = {expected:>2} | {'PASS' if got == expected else 'FAIL'}")

brute force =  7 | expected =  7 | PASS
brute force =  5 | expected =  5 | PASS
brute force = -1 | expected = -1 | PASS


## Brute force tốn bao nhiêu công?

Từ mỗi đỉnh `s`, DFS thăm **cả `n` đỉnh**. Có `n` đỉnh xuất phát → tổng cộng `n · n = n²` lượt thăm.

- `n = 1 000` → `10^6` lượt: ổn.
- `n = 10^5` → `10^10` lượt: Python làm khoảng `10^7` thao tác đơn giản mỗi giây → cần cỡ **hàng nghìn giây**. Chắc chắn **TLE** (Time Limit Exceeded).

Cell dưới đếm số lượt thăm thật trên các cây ngẫu nhiên và so với `n²`.

In [13]:
import random

def random_tree(n, max_cost=10, max_value=15, seed=None):
    """Sinh cây ngẫu nhiên: đỉnh i (i >= 1) nối với một đỉnh ngẫu nhiên đứng trước nó."""
    rng = random.Random(seed)
    edges = [[rng.randrange(i), i, rng.randint(1, max_cost)] for i in range(1, n)]
    values = [rng.randint(1, max_value) for _ in range(n)]
    return edges, values

def count_bruteforce_visits(edges, values):
    n = len(values)
    adj = build_adj(n, edges)
    visits = 0
    for s in range(n):
        stack = [(s, -1)]
        while stack:
            u, par = stack.pop()
            visits += 1
            for v, _ in adj[u]:
                if v != par:
                    stack.append((v, u))
    return visits

for n in [10, 100, 500]:
    e, v = random_tree(n, seed=n)
    print(f"n = {n:>4}: số lượt thăm = {count_bruteforce_visits(e, v):>7,}  (n² = {n*n:,})")
print(f"n = 100000: số lượt thăm ước tính = {10**5 * 10**5:,}  -> quá chậm")

n =   10: số lượt thăm =     100  (n² = 100)
n =  100: số lượt thăm =  10,000  (n² = 10,000)
n =  500: số lượt thăm = 250,000  (n² = 250,000)
n = 100000: số lượt thăm ước tính = 10,000,000,000  -> quá chậm


# 5. Bước đột phá 1 — Chỉ cần biết chẵn/lẻ, không cần XOR thật

## Câu hỏi

Đề chỉ hỏi "XOR có **số bit 1 chẵn** không?". Liệu ta có thể biết điều đó **mà không cần tính giá trị XOR**, chỉ từ chẵn/lẻ số bit 1 của từng giá trị?

## Suy luận theo từng vị trí bit

Xét hai số `a`, `b`. Nhìn từng vị trí bit:

- Nếu đúng **một** trong hai số có bit `1` ở vị trí đó → XOR có bit `1`. Số bit 1 "chuyển" nguyên sang kết quả.
- Nếu **cả hai** đều có bit `1` → XOR có bit `0`. Ta **mất 2** bit 1 cùng lúc.
- Nếu cả hai đều `0` → không có gì.

Vậy so với tổng `popcount(a) + popcount(b)`, kết quả XOR chỉ có thể **bớt đi một số chẵn** bit 1. Mà bớt đi số chẵn thì **không đổi chẵn/lẻ**!

Thí nghiệm với vài cặp số:

In [14]:
pairs = [(3, 1), (3, 7), (1, 7), (5, 6), (12, 10), (255, 1)]
print(" a    b  | pc(a) pc(b) | a^b  pc(a^b) | (pc(a)+pc(b))%2  pc(a^b)%2")
for a, b in pairs:
    pa, pb, px = a.bit_count(), b.bit_count(), (a ^ b).bit_count()
    print(f"{a:>3} {b:>4}  |   {pa}     {pb}   | {a ^ b:>3}     {px}    |        {(pa + pb) % 2}               {px % 2}")

 a    b  | pc(a) pc(b) | a^b  pc(a^b) | (pc(a)+pc(b))%2  pc(a^b)%2
  3    1  |   2     1   |   2     1    |        1               1
  3    7  |   2     3   |   4     1    |        1               1
  1    7  |   1     3   |   6     2    |        0               0
  5    6  |   2     2   |   3     2    |        0               0
 12   10  |   2     2   |   6     2    |        0               0
255    1  |   8     1   | 254     7    |        1               1


Hai cột cuối **luôn bằng nhau**. Công thức chính xác đứng sau hiện tượng này:

```text
popcount(a ^ b) = popcount(a) + popcount(b) - 2 · popcount(a & b)
```

(`a & b` là các vị trí mà **cả hai** cùng có bit 1 — mỗi vị trí như vậy làm mất 2 bit.) Lấy `% 2` hai vế, phần `2 · (...)` biến mất:

```text
popcount(a ^ b) % 2 = (popcount(a) + popcount(b)) % 2 = parity(a) ^ parity(b)
```

Áp dụng nhiều lần cho cả một đường đi `v1, v2, ..., vk`:

```text
parity(v1 ^ v2 ^ ... ^ vk) = parity(v1) ^ parity(v2) ^ ... ^ parity(vk)
```

Cell dưới kiểm tra công thức trên **mọi** cặp `a, b < 256` và trên các danh sách ngẫu nhiên.

In [15]:
for a in range(256):
    for b in range(256):
        assert (a ^ b).bit_count() == a.bit_count() + b.bit_count() - 2 * (a & b).bit_count()
        assert ((a ^ b).bit_count() & 1) == ((a.bit_count() & 1) ^ (b.bit_count() & 1))
print("Đúng cho mọi cặp a, b < 256")

rng = random.Random(0)
for _ in range(2000):
    nums = [rng.randint(1, 10**9) for _ in range(rng.randint(2, 8))]
    x, par = 0, 0
    for v in nums:
        x ^= v
        par ^= v.bit_count() & 1
    assert (x.bit_count() & 1) == par
print("Đúng cho 2000 danh sách ngẫu nhiên (giá trị tới 10^9)")

Đúng cho mọi cặp a, b < 256
Đúng cho 2000 danh sách ngẫu nhiên (giá trị tới 10^9)


## Bài toán mới, gọn hơn hẳn

Gán cho mỗi đỉnh một bit:

```text
p[i] = parity(values[i]) = values[i].bit_count() & 1
```

- `p[i] = 0`: ta gọi là **đỉnh chẵn** (giá trị có số bit 1 chẵn).
- `p[i] = 1`: ta gọi là **đỉnh lẻ**.

Khi đó:

> Một đường đi là Balanced-XOR **khi và chỉ khi** XOR các bit `p` trên đường bằng `0`, tức **số đỉnh lẻ trên đường là số chẵn**.

Giá trị `values[i]` thật (tới `10^9`) không còn quan trọng nữa — chỉ còn **một bit cho mỗi đỉnh**. Ta gọi XOR các `p` trên một đường là **parity của đường**.

In [16]:
def parities(values):
    return [v.bit_count() & 1 for v in values]

for k, (edges, values, expected) in enumerate(EXAMPLES, start=1):
    print(f"Ví dụ {k}: values = {values} -> p = {parities(values)}")

# Kiểm tra lại 10 đường của Ví dụ 1, chỉ dùng bit p
p1 = parities([3, 1, 7, 1, 3])
for path in all_paths:
    par = 0
    for node in path:
        par ^= p1[node]
    print(" - ".join(map(str, path)).ljust(14), "bits =", [p1[x] for x in path], "-> parity", par, "(hợp lệ)" if par == 0 else "")

Ví dụ 1: values = [3, 1, 7, 1, 3] -> p = [0, 1, 1, 1, 0]
Ví dụ 2: values = [1, 7, 3] -> p = [1, 1, 0]
Ví dụ 3: values = [1, 3, 5, 9] -> p = [1, 0, 0, 0]
0 - 1          bits = [0, 1] -> parity 1 
0 - 2          bits = [0, 1] -> parity 1 
0 - 3          bits = [0, 1] -> parity 1 
0 - 3 - 4      bits = [0, 1, 0] -> parity 1 
1 - 0 - 2      bits = [1, 0, 1] -> parity 0 (hợp lệ)
1 - 0 - 3      bits = [1, 0, 1] -> parity 0 (hợp lệ)
1 - 0 - 3 - 4  bits = [1, 0, 1, 0] -> parity 0 (hợp lệ)
2 - 0 - 3      bits = [1, 0, 1] -> parity 0 (hợp lệ)
2 - 0 - 3 - 4  bits = [1, 0, 1, 0] -> parity 0 (hợp lệ)
3 - 4          bits = [1, 0] -> parity 1 


# 6. Bước đột phá 2 — Đường rẻ nhất chỉ có 2 hoặc 3 đỉnh

## Câu hỏi

Ở Ví dụ 1, đường rẻ nhất có 3 đỉnh. Ở Ví dụ 2, đường rẻ nhất có 2 đỉnh. Có khi nào đường rẻ nhất cần 4, 5, ... đỉnh không?

## Mấu chốt: chi phí cạnh luôn dương

Vì `c_i >= 1`, nếu một đường hợp lệ **chứa bên trong** một **đoạn con** (sub-path: một khúc liên tiếp của đường, có ít nhất 2 đỉnh) cũng hợp lệ, thì đoạn con đó **rẻ hơn hoặc bằng** (bỏ bớt cạnh chỉ làm tổng giảm). Vậy chỉ cần trả lời:

> Mọi đường hợp lệ có luôn chứa một đoạn con hợp lệ **ngắn** không?

Viết các bit `p` dọc theo đường: `x1, x2, ..., xk` (k >= 2), với XOR tất cả bằng 0.

**Trường hợp A — có hai đỉnh kề nhau cùng bit** (`x_i == x_{i+1}`): đoạn con 2 đỉnh `x_i, x_{i+1}` có XOR = 0 → hợp lệ. Đó là **một cạnh có hai đầu cùng parity**.

**Trường hợp B — các bit luân phiên** `0,1,0,1,...` hoặc `1,0,1,0,...`:

- `k = 2`: dãy `01`/`10` có XOR = 1 → không hợp lệ, nên trường hợp này không xảy ra.
- `k = 3`: `010` có XOR 1 (không hợp lệ), `101` có XOR 0 (hợp lệ) → chính nó là đường 3 đỉnh `1 - 0 - 1`.
- `k >= 4`: dãy luân phiên dài ≥ 4 luôn chứa cửa sổ `1, 0, 1` → đoạn con 3 đỉnh hợp lệ.

**Kết luận:** mọi đường hợp lệ đều chứa một đoạn con hợp lệ có **2 đỉnh** (cạnh cùng parity) hoặc **3 đỉnh** dạng `lẻ - chẵn - lẻ`. Vì vậy **đường hợp lệ rẻ nhất luôn có 2 hoặc 3 đỉnh**.

## Bảng 8 mẫu đường 3 đỉnh (đầu - giữa - đầu)

| Mẫu bit | XOR | Hợp lệ? | Có chứa cạnh cùng parity (rẻ hơn)? |
|---|:---:|:---:|---|
| 0-0-0 | 0 | có | có (0-0) |
| 0-0-1 | 1 | không | — |
| 0-1-0 | 1 | không | — |
| 0-1-1 | 0 | có | có (1-1) |
| 1-0-0 | 1 | không | — |
| **1-0-1** | 0 | **có** | **không** → phải xét riêng |
| 1-1-0 | 0 | có | có (1-1) |
| 1-1-1 | 1 | không | — |

Chỉ có mẫu `1-0-1` (**đỉnh giữa chẵn, hai đầu lẻ**) là đường 3 đỉnh "thật sự cần thiết".

In [17]:
from itertools import product

def has_short_balanced_window(bits):
    """Có đoạn liên tiếp dài 2 hoặc 3 với XOR = 0 không?"""
    k = len(bits)
    for length in (2, 3):
        for i in range(k - length + 1):
            w = bits[i:i + length]
            x = 0
            for b in w:
                x ^= b
            if x == 0:
                return True
    return False

checked = 0
for k in range(2, 13):
    for bits in product([0, 1], repeat=k):      # mọi dãy bit độ dài k
        total = 0
        for b in bits:
            total ^= b
        if total == 0:                           # đường hợp lệ
            assert has_short_balanced_window(bits), bits
            checked += 1
print("Đã kiểm tra", checked, "dãy bit hợp lệ (độ dài 2..12): dãy nào cũng chứa đoạn con hợp lệ 2 hoặc 3 đỉnh")

seq = [0, 1, 0, 1, 0]      # một đường 5 đỉnh luân phiên, XOR = 0
print("Ví dụ:", seq, "XOR =", 0 ^ 1 ^ 0 ^ 1 ^ 0, "-> cửa sổ", seq[1:4], "XOR =", seq[1] ^ seq[2] ^ seq[3])

Đã kiểm tra 4094 dãy bit hợp lệ (độ dài 2..12): dãy nào cũng chứa đoạn con hợp lệ 2 hoặc 3 đỉnh
Ví dụ: [0, 1, 0, 1, 0] XOR = 0 -> cửa sổ [1, 0, 1] XOR = 0


Thí nghiệm thứ hai, trực tiếp trên cây: sinh nhiều cây ngẫu nhiên nhỏ, liệt kê **mọi** đường hợp lệ, tìm chi phí nhỏ nhất và đếm xem các đường **đạt chi phí nhỏ nhất** có bao nhiêu đỉnh.

Theo lý luận trên (chi phí **dương thật sự**), một đường dài hơn luôn chứa đoạn con hợp lệ **rẻ hơn hẳn**, nên mọi đường tối ưu phải có đúng 2 hoặc 3 đỉnh.

In [18]:
from collections import Counter

def all_balanced_paths(edges, values):
    """Trả về list (cost, số_đỉnh) của mọi đường hợp lệ (mỗi đường đếm 1 lần: s < t)."""
    n = len(values)
    adj = build_adj(n, edges)
    out = []
    for s in range(n):
        stack = [(s, -1, values[s], 0, 1)]
        while stack:
            u, par, x, cost, cnt = stack.pop()
            if u > s and x.bit_count() % 2 == 0:
                out.append((cost, cnt))
            for v, c in adj[u]:
                if v != par:
                    stack.append((v, u, x ^ values[v], cost + c, cnt + 1))
    return out

lengths = Counter()
for seed in range(300):
    rng = random.Random(seed)
    e, v = random_tree(rng.randint(2, 9), max_cost=5, seed=seed)
    paths = all_balanced_paths(e, v)
    if paths:
        best = min(cost for cost, _ in paths)
        for cost, cnt in paths:
            if cost == best:
                lengths[cnt] += 1
print("Số đỉnh của các đường tối ưu:", dict(sorted(lengths.items())))

Số đỉnh của các đường tối ưu: {2: 320, 3: 19}


## Hệ quả: công thức đáp án

```text
A = chi phí nhỏ nhất của một CẠNH (u, v) có p[u] == p[v]
B = min trên mọi đỉnh CHẴN m (p[m] = 0) của:
        tổng 2 chi phí nhỏ nhất trong các cạnh nối m với hàng xóm LẺ
đáp án = min(A, B)   (nếu cả hai đều không tồn tại → -1)
```

Kiểm tra lại bằng tay trên các ví dụ (`p` đã tính ở mục 5):

- **Ví dụ 1**, `p = [0,1,1,1,0]`: không cạnh nào cùng parity → `A` không có. Đỉnh chẵn `0` có 3 hàng xóm lẻ với chi phí `4, 6, 3` → hai số nhỏ nhất `3 + 4 = 7`. Đỉnh chẵn `4` chỉ có 1 hàng xóm. → đáp án `7`. ✔
- **Ví dụ 2**, `p = [1,1,0]`: cạnh `0-1` cùng parity, chi phí `5` → `A = 5`. Đỉnh chẵn `2` chỉ có 1 hàng xóm. → đáp án `5`. ✔
- **Ví dụ 3**, `p = [1,0,0,0]`: không cạnh nào cùng parity; ba đỉnh chẵn `1, 2, 3` mỗi đỉnh chỉ có 1 hàng xóm → không có gì → `-1`. ✔

# 7. Cách 2 — Duyệt mọi đường 2–3 đỉnh (cách ngây thơ)

## Ý tưởng

Dùng ngay Bước đột phá 2 theo cách thẳng thắn nhất:

1. Đường 2 đỉnh = một cạnh. Duyệt mọi cạnh `[u, v, c]`, nếu `p[u] == p[v]` thì cập nhật đáp án bằng `c`.
2. Đường 3 đỉnh `a - m - b` = chọn đỉnh giữa `m` và **một cặp** hàng xóm khác nhau `a, b` của `m`. Duyệt mọi `m`, mọi cặp `(i, j)` với `i < j` trong `adj[m]`; nếu `p[a] ^ p[m] ^ p[b] == 0` thì cập nhật bằng `c_a + c_b`.

Cách này chưa cần tối ưu gì, cứ kiểm tra **đúng điều kiện parity** của đường.

In [19]:
def min_cost_short_naive(edges: List[List[int]], values: List[int]) -> int:
    n = len(values)
    adj = build_adj(n, edges)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    ans = INF
    # (1) đường 2 đỉnh
    for u, v, c in edges:
        if p[u] == p[v]:
            ans = min(ans, c)
    # (2) đường 3 đỉnh a - m - b
    for m in range(n):
        nb = adj[m]
        for i in range(len(nb)):
            for j in range(i + 1, len(nb)):
                a, ca = nb[i]
                b, cb = nb[j]
                if p[a] ^ p[m] ^ p[b] == 0:
                    ans = min(ans, ca + cb)
    return -1 if ans == INF else ans

for edges, values, expected in EXAMPLES:
    got = min_cost_short_naive(edges, values)
    print(f"short naive = {got:>2} | expected = {expected:>2} | {'PASS' if got == expected else 'FAIL'}")

short naive =  7 | expected =  7 | PASS
short naive =  5 | expected =  5 | PASS
short naive = -1 | expected = -1 | PASS


## Cách 2 có đủ nhanh không?

Phần (1) tốn `O(n)`. Phần (2), đỉnh `m` có `deg(m)` hàng xóm thì có `deg(m)·(deg(m)-1)/2` cặp. Tổng công việc là `O(n + Σ deg(m)²)`.

- Cây ngẫu nhiên, cây đường thẳng: bậc nhỏ → rất nhanh.
- **Cây hình sao** (một đỉnh tâm nối với tất cả `n - 1` đỉnh còn lại): tâm có bậc `n - 1` → khoảng `n²/2` cặp ≈ **5 tỷ** khi `n = 10^5` → TLE.

Cell dưới tính số cặp hàng xóm cho hai hình dạng cây (chỉ đếm bằng công thức, không chạy thuật toán).

In [20]:
def neighbor_pairs(n, edges):
    deg = [0] * n
    for u, v, _ in edges:
        deg[u] += 1
        deg[v] += 1
    return sum(d * (d - 1) // 2 for d in deg)

N = 10**5
star_edges = [[0, i, 1] for i in range(1, N)]          # hình sao, tâm 0
line_edges = [[i, i + 1, 1] for i in range(N - 1)]     # đường thẳng
print(f"Cây đường thẳng: {neighbor_pairs(N, line_edges):>15,} cặp hàng xóm")
print(f"Cây hình sao   : {neighbor_pairs(N, star_edges):>15,} cặp hàng xóm  -> quá chậm!")

Cây đường thẳng:          99,998 cặp hàng xóm
Cây hình sao   :   4,999,850,001 cặp hàng xóm  -> quá chậm!


# 8. Cách 3 — `O(n)`: chỉ giữ 2 cạnh rẻ nhất cho mỗi đỉnh chẵn

## Vấn đề của Cách 2

Ở một đỉnh có bậc lớn, Cách 2 thử **mọi cặp** hàng xóm. Nhưng theo bảng 8 mẫu ở mục 6, đường 3 đỉnh duy nhất cần xét là `lẻ - chẵn - lẻ`. Với một đỉnh giữa chẵn `m` cố định, trong tất cả các cặp hàng xóm lẻ, cặp **rẻ nhất** chắc chắn là **hai cạnh rẻ nhất** tới hàng xóm lẻ. Các cặp khác không bao giờ thắng được → không cần thử.

## Mấu chốt: mỗi cạnh chỉ cần nhìn một lần

Duyệt từng cạnh `[u, v, c]`:

- Nếu `p[u] == p[v]`: đây là đường 2 đỉnh hợp lệ → `ans = min(ans, c)`.
- Nếu `p[u] != p[v]`: một đầu chẵn, một đầu lẻ. Gọi `m` là **đầu chẵn**. Cạnh này là "cạnh từ đỉnh chẵn `m` tới hàng xóm lẻ" → đưa `c` vào bộ "2 giá trị nhỏ nhất" `best1[m], best2[m]` (mục 3.7).

Sau khi duyệt hết cạnh: với mỗi đỉnh `m` mà `best2[m] < INF` (có ít nhất 2 hàng xóm lẻ), cập nhật `ans = min(ans, best1[m] + best2[m])`.

Không cần danh sách kề, không cần DFS — chỉ một vòng lặp qua `edges` và một vòng qua các đỉnh!

In [21]:
def min_cost_local(edges: List[List[int]], values: List[int]) -> int:
    n = len(values)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    best1 = [INF] * n          # cạnh rẻ nhất từ đỉnh chẵn m tới hàng xóm lẻ
    best2 = [INF] * n          # cạnh rẻ thứ hai
    ans = INF
    for u, v, c in edges:
        if p[u] == p[v]:                     # đường 2 đỉnh hợp lệ
            if c < ans:
                ans = c
        else:                                # một đầu chẵn, một đầu lẻ
            m = u if p[u] == 0 else v        # m = đầu chẵn (đỉnh giữa tiềm năng)
            if c < best1[m]:
                best2[m] = best1[m]
                best1[m] = c
            elif c < best2[m]:
                best2[m] = c
    for m in range(n):                       # đường 3 đỉnh lẻ - m - lẻ
        if best2[m] < INF:
            ans = min(ans, best1[m] + best2[m])
    return -1 if ans == INF else ans

for edges, values, expected in EXAMPLES:
    got = min_cost_local(edges, values)
    print(f"local O(n) = {got:>2} | expected = {expected:>2} | {'PASS' if got == expected else 'FAIL'}")

local O(n) =  7 | expected =  7 | PASS
local O(n) =  5 | expected =  5 | PASS
local O(n) = -1 | expected = -1 | PASS


## Độ phức tạp của Cách 3

- Tính `p`: `n` lần `bit_count()` — mỗi lần làm việc trên số ≤ 30 bit, coi như `O(1)`.
- Vòng lặp cạnh: `n - 1` cạnh, mỗi cạnh `O(1)`.
- Vòng lặp đỉnh: `n` đỉnh, mỗi đỉnh `O(1)`.

→ **Thời gian `O(n)`, bộ nhớ `O(n)`** (các mảng `p`, `best1`, `best2`). Cây hình sao không còn là vấn đề: tâm sao chỉ cập nhật `best1/best2` mỗi cạnh một lần.

# 9. Mô phỏng Cách 3 (trace từng bước)

Ta dùng một ví dụ **tự tạo** có đủ các nhánh đáng chú ý (cạnh cùng parity, đẩy `best1` xuống `best2`, chen vào `best2`):

```text
edges  = [[1,3,8],[0,1,2],[3,4,1],[1,2,3],[3,5,9]]
values = [7, 5, 4, 1, 6, 8]
popcount = [3, 2, 1, 1, 2, 1]   ->   p = [1, 0, 1, 1, 0, 1]
```

```text
 (0) lẻ --2-- (1) chẵn --3-- (2) lẻ
               |
               8
               |
 (4) chẵn --1-- (3) lẻ --9-- (5) lẻ
```

Bảng trạng thái mong đợi (chỉ ghi `best1/best2` của đỉnh vừa thay đổi):

| Bước | Cạnh (u, v, c) | p[u], p[v] | Loại | Hành động | Trạng thái sau | ans |
|---|---|---|---|---|---|---|
| 1 | (1, 3, 8) | 0, 1 | khác parity, m = 1 | 8 < best1[1] = ∞ | best1[1] = 8, best2[1] = ∞ | ∞ |
| 2 | (0, 1, 2) | 1, 0 | khác parity, m = 1 | 2 < best1[1] = 8 → đẩy xuống | best1[1] = 2, best2[1] = 8 | ∞ |
| 3 | (3, 4, 1) | 1, 0 | khác parity, m = 4 | 1 < best1[4] = ∞ | best1[4] = 1, best2[4] = ∞ | ∞ |
| 4 | (1, 2, 3) | 0, 1 | khác parity, m = 1 | 3 ≥ 2 nhưng 3 < 8 → chen | best1[1] = 2, best2[1] = 3 | ∞ |
| 5 | (3, 5, 9) | 1, 1 | **cùng parity** | ans = min(∞, 9) | — | 9 |
| cuối | đỉnh 1 | | đường 3 đỉnh | 2 + 3 = 5 < 9 | | **5** |
| cuối | đỉnh 4 | | chỉ 1 hàng xóm lẻ | bỏ qua | | 5 |

Đáp án `5` ứng với đường `0 - 1 - 2` (bits `1-0-1`), rẻ hơn cạnh cùng parity `3 - 5` (chi phí 9).

In [22]:
def trace_min_cost_local(edges, values):
    n = len(values)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    show = lambda x: "∞" if x == INF else str(x)
    best1, best2 = [INF] * n, [INF] * n
    ans = INF
    print("p =", p)
    for step, (u, v, c) in enumerate(edges, start=1):
        if p[u] == p[v]:
            ans = min(ans, c)
            print(f"Bước {step}: cạnh ({u},{v},{c}) cùng parity -> ans = {show(ans)}")
        else:
            m = u if p[u] == 0 else v
            if c < best1[m]:
                best2[m], best1[m] = best1[m], c
                act = "đẩy/đặt best1"
            elif c < best2[m]:
                best2[m] = c
                act = "chen vào best2"
            else:
                act = "bỏ qua"
            print(f"Bước {step}: cạnh ({u},{v},{c}) khác parity, m={m}: {act:<14}"
                  f" -> best1[{m}]={show(best1[m])}, best2[{m}]={show(best2[m])}")
    for m in range(n):
        if best2[m] < INF:
            cand = best1[m] + best2[m]
            ans = min(ans, cand)
            print(f"Cuối: đỉnh chẵn {m}: best1 + best2 = {best1[m]} + {best2[m]} = {cand} -> ans = {show(ans)}")
    result = -1 if ans == INF else ans
    print("Kết quả:", result)
    return result

TRACE_EDGES = [[1, 3, 8], [0, 1, 2], [3, 4, 1], [1, 2, 3], [3, 5, 9]]
TRACE_VALUES = [7, 5, 4, 1, 6, 8]
trace_min_cost_local(TRACE_EDGES, TRACE_VALUES)
print("Brute force xác nhận:", min_cost_bruteforce(TRACE_EDGES, TRACE_VALUES))

p = [1, 0, 1, 1, 0, 1]
Bước 1: cạnh (1,3,8) khác parity, m=1: đẩy/đặt best1  -> best1[1]=8, best2[1]=∞
Bước 2: cạnh (0,1,2) khác parity, m=1: đẩy/đặt best1  -> best1[1]=2, best2[1]=8
Bước 3: cạnh (3,4,1) khác parity, m=4: đẩy/đặt best1  -> best1[4]=1, best2[4]=∞
Bước 4: cạnh (1,2,3) khác parity, m=1: chen vào best2 -> best1[1]=2, best2[1]=3
Bước 5: cạnh (3,5,9) cùng parity -> ans = 9
Cuối: đỉnh chẵn 1: best1 + best2 = 2 + 3 = 5 -> ans = 5
Kết quả: 5
Brute force xác nhận: 5


Output khớp với bảng ở trên (đỉnh `4` không được in ở phần "Cuối" vì `best2[4] = ∞`, tức bị bỏ qua). Thử thêm trên **Ví dụ 1** (đáp án đến từ đường 3 đỉnh) và **Ví dụ 3** (trường hợp `-1`): ở Ví dụ 3, đỉnh `0` là đỉnh **lẻ** nên mọi cạnh đều đưa chi phí vào `best1` của các lá chẵn `1, 2, 3` — mỗi lá chỉ nhận được **một** cạnh, nên `best2` mãi là `∞`.

In [23]:
trace_min_cost_local(EXAMPLES[0][0], EXAMPLES[0][1])
print("-" * 60)
trace_min_cost_local(EXAMPLES[2][0], EXAMPLES[2][1])

p = [0, 1, 1, 1, 0]
Bước 1: cạnh (0,1,4) khác parity, m=0: đẩy/đặt best1  -> best1[0]=4, best2[0]=∞
Bước 2: cạnh (0,2,6) khác parity, m=0: chen vào best2 -> best1[0]=4, best2[0]=6
Bước 3: cạnh (0,3,3) khác parity, m=0: đẩy/đặt best1  -> best1[0]=3, best2[0]=4
Bước 4: cạnh (3,4,5) khác parity, m=4: đẩy/đặt best1  -> best1[4]=5, best2[4]=∞
Cuối: đỉnh chẵn 0: best1 + best2 = 3 + 4 = 7 -> ans = 7
Kết quả: 7
------------------------------------------------------------
p = [1, 0, 0, 0]
Bước 1: cạnh (0,1,5) khác parity, m=1: đẩy/đặt best1  -> best1[1]=5, best2[1]=∞
Bước 2: cạnh (0,2,7) khác parity, m=2: đẩy/đặt best1  -> best1[2]=7, best2[2]=∞
Bước 3: cạnh (0,3,9) khác parity, m=3: đẩy/đặt best1  -> best1[3]=9, best2[3]=∞
Kết quả: -1


-1

# 10. Cách 4 — Tree DP với 2 trạng thái chẵn/lẻ (kỹ thuật tổng quát)

## Vì sao vẫn học thêm cách này?

Cách 3 rất gọn, nhưng nó dựa **hoàn toàn** vào Bước đột phá 2 ("đường rẻ nhất rất ngắn"). Chỉ cần đề đổi một chút — ví dụ hỏi chi phí **lớn nhất**, hoặc chi phí cạnh có thể **âm** — thì đường tốt nhất có thể dài tùy ý và Cách 3 sụp đổ.

**Tree DP** (Dynamic Programming on trees) chỉ dùng Bước đột phá 1 (parity) và xử lý được mọi đường đi, dài bao nhiêu cũng được. Đây là kỹ thuật chuẩn cho rất nhiều bài Hard về "đường đi trên cây".

## Bước 1: mỗi đường có một "đỉnh cao nhất"

Treo cây tại gốc `0` (mục 3.8). Với một đường `u ... v`, đỉnh **gần gốc nhất** trên đường gọi là **đỉnh cao nhất** `t` (chính là LCA — Lowest Common Ancestor, tổ tiên chung thấp nhất). Từ `t`, đường **rẽ xuống hai phía**:

```text
            t                 Ví dụ 1 (gốc 0):
          /   \                 đường 1-0-3-4: t = 0, nhánh trái 0-1, nhánh phải 0-3-4
       ...     ...              đường 3-4    : t = 3, nhánh trái chỉ là [3], nhánh phải 3-4
        u       v
```

Một trong hai nhánh có thể chỉ gồm mình `t` (khi `t` là một đầu mút). Ý tưởng: **đếm mỗi đường đúng một lần, tại đỉnh cao nhất của nó**.

## Bước 2: định nghĩa trạng thái

```text
down[u][q] = chi phí nhỏ nhất của một "nhánh đi xuống" bắt đầu tại u,
             kết thúc ở một đỉnh trong cây con của u (có thể là chính u, chi phí 0),
             sao cho XOR các bit p trên nhánh (TÍNH CẢ u) bằng q        (q ∈ {0, 1})
```

Khởi tạo (nhánh chỉ gồm `u`): `down[u][p[u]] = 0`, ô còn lại `= INF`.

## Bước 3: gộp từng con `v` vào cha `u` (cạnh chi phí `c`)

**(a) Nhánh mới đi qua `v`:** lấy nhánh tốt nhất của `v` có parity `q`, nối thêm `u` ở đầu: parity thành `q ^ p[u]`, chi phí thêm `c`:

```text
cand[q ^ p[u]] = down[v][q] + c
```

**(b) Ghép hai nhánh thành đường đi:** một nhánh cũ `down[u][a]` (đi vào các con đã gộp trước đó, hoặc chỉ là `u`) và một nhánh mới `cand[b]`. Cả hai nhánh đều chứa `u`, nên `u` bị đếm **hai lần** — XOR hai lần thì triệt tiêu, phải XOR lại `p[u]` một lần:

```text
parity đường = a ^ b ^ p[u]   → cần bằng 0   → b = a ^ p[u]
ans = min(ans, down[u][a] + cand[a ^ p[u]])     với a = 0, 1
```

Ví dụ số (Ví dụ 1, `u = 0`, `p[0] = 0`): nhánh cũ `0-1` có parity `0^1 = 1`, chi phí 4; nhánh mới `0-3` có parity `0^1 = 1`, chi phí 3. Đường `1-0-3` có parity `1 ^ 1 ^ 0 = 0` ✔, chi phí `4 + 3 = 7` ✔.

Vì `cand` luôn có ít nhất một cạnh, đường ghép được luôn có **≥ 2 đỉnh phân biệt** — không bao giờ đếm nhầm "đường" chỉ gồm một đỉnh.

**(c) Cập nhật:** `down[u][q] = min(down[u][q], cand[q])`.

**Thứ tự (b) trước (c) là bắt buộc!** Nếu cập nhật trước rồi mới ghép, ta có thể ghép nhánh qua `v` **với chính nó** → "đường" `v - u - v` không có thật (xem mục Lỗi thường gặp).

## Bước 4: thứ tự xử lý

Khi gộp `v` vào `u`, `down[v]` phải **đã hoàn chỉnh** (mọi con của `v` đã gộp xong). Duyệt `reversed(order)` (mục 3.8) đảm bảo điều đó.

In [24]:
def min_cost_tree_dp(edges: List[List[int]], values: List[int]) -> int:
    n = len(values)
    adj = build_adj(n, edges)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    parent, parent_cost, order = root_tree(adj, 0)     # mục 3.8

    down = [[INF, INF] for _ in range(n)]
    for u in range(n):
        down[u][p[u]] = 0                               # nhánh chỉ gồm u
    ans = INF

    for v in reversed(order):                           # con trước, cha sau
        u = parent[v]
        if u == -1:                                     # gốc: không có cha để gộp vào
            continue
        c = parent_cost[v]
        # (a) các nhánh mới từ u đi xuống qua v
        cand = [INF, INF]
        for q in (0, 1):
            if down[v][q] + c < cand[q ^ p[u]]:
                cand[q ^ p[u]] = down[v][q] + c
        # (b) ghép nhánh cũ của u với nhánh mới (TRƯỚC khi cập nhật)
        for a in (0, 1):
            total = down[u][a] + cand[a ^ p[u]]
            if total < ans:
                ans = total
        # (c) cập nhật down[u]
        for q in (0, 1):
            if cand[q] < down[u][q]:
                down[u][q] = cand[q]
    return -1 if ans == INF else ans

for edges, values, expected in EXAMPLES:
    got = min_cost_tree_dp(edges, values)
    print(f"tree DP = {got:>2} | expected = {expected:>2} | {'PASS' if got == expected else 'FAIL'}")

tree DP =  7 | expected =  7 | PASS
tree DP =  5 | expected =  5 | PASS
tree DP = -1 | expected = -1 | PASS


## Mô phỏng Cách 4 trên Ví dụ 1

`p = [0, 1, 1, 1, 0]`, gốc `0`. Ở mục 3.8 ta đã có `reversed(order) = [1, 2, 4, 3, 0]`, nên thứ tự gộp là: `1→0`, `2→0`, `4→3`, `3→0` (đỉnh `0` là gốc, bỏ qua).

Khởi tạo: `down[0] = [0, ∞]`, `down[1] = down[2] = down[3] = [∞, 0]`, `down[4] = [0, ∞]`.

| Gộp | c | down[v] | cand | Ghép a=0 / a=1 | ans | down[u] sau |
|---|---:|---|---|---|---:|---|
| 1 → 0 | 4 | [∞, 0] | [∞, 4] | ∞ / ∞ + 4 = ∞ | ∞ | down[0] = [0, 4] |
| 2 → 0 | 6 | [∞, 0] | [∞, 6] | 0 + ∞ / 4 + 6 = **10** | 10 | down[0] = [0, 4] |
| 4 → 3 | 5 | [0, ∞] | [∞, 5] | ∞ + 5 / 0 + ∞ | 10 | down[3] = [∞, 0] |
| 3 → 0 | 3 | [∞, 0] | [∞, 3] | 0 + ∞ / 4 + 3 = **7** | 7 | down[0] = [0, 3] |

Đọc bảng: khi gộp `2` vào `0`, ta tìm thấy đường `1-0-2` (chi phí 10). Khi gộp `3` vào `0`, nhánh cũ `0-1` (parity 1, chi phí 4) ghép với nhánh mới `0-3` (parity 1, chi phí 3) → đường `1-0-3`, chi phí `7`.

Ở bước `4 → 3`, `p[3] = 1` nên điều kiện ghép là `b = a ^ 1`: hai cặp được xét là `down[3][0] + cand[1] = ∞ + 5` và `down[3][1] + cand[0] = 0 + ∞`, đều `∞` — đúng vì đường `3-4` có bits `1-0`, không hợp lệ.

In [25]:
def trace_min_cost_tree_dp(edges, values):
    n = len(values)
    adj = build_adj(n, edges)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    show = lambda arr: "[" + ", ".join("∞" if x == INF else str(x) for x in arr) + "]"
    parent, parent_cost, order = root_tree(adj, 0)
    down = [[INF, INF] for _ in range(n)]
    for u in range(n):
        down[u][p[u]] = 0
    ans = INF
    print("p =", p, "| reversed(order) =", list(reversed(order)))
    for v in reversed(order):
        u = parent[v]
        if u == -1:
            continue
        c = parent_cost[v]
        cand = [INF, INF]
        for q in (0, 1):
            cand[q ^ p[u]] = min(cand[q ^ p[u]], down[v][q] + c)
        pieces = []
        for a in (0, 1):
            total = down[u][a] + cand[a ^ p[u]]
            pieces.append("∞" if total == INF else str(total))
            ans = min(ans, total)
        before = show(down[u])
        for q in (0, 1):
            down[u][q] = min(down[u][q], cand[q])
        print(f"Gộp {v} -> {u} (c={c}): down[{v}]={show(down[v])}, cand={show(cand)}, "
              f"ghép a=0/a=1: {pieces[0]}/{pieces[1]}, ans={'∞' if ans == INF else ans}, "
              f"down[{u}]: {before} -> {show(down[u])}")
    result = -1 if ans == INF else ans
    print("Kết quả:", result)
    return result

trace_min_cost_tree_dp(EXAMPLES[0][0], EXAMPLES[0][1])

p = [0, 1, 1, 1, 0] | reversed(order) = [1, 2, 4, 3, 0]
Gộp 1 -> 0 (c=4): down[1]=[∞, 0], cand=[∞, 4], ghép a=0/a=1: ∞/∞, ans=∞, down[0]: [0, ∞] -> [0, 4]
Gộp 2 -> 0 (c=6): down[2]=[∞, 0], cand=[∞, 6], ghép a=0/a=1: ∞/10, ans=10, down[0]: [0, 4] -> [0, 4]
Gộp 4 -> 3 (c=5): down[4]=[0, ∞], cand=[∞, 5], ghép a=0/a=1: ∞/∞, ans=10, down[3]: [∞, 0] -> [∞, 0]
Gộp 3 -> 0 (c=3): down[3]=[∞, 0], cand=[∞, 3], ghép a=0/a=1: ∞/7, ans=7, down[0]: [0, 4] -> [0, 3]
Kết quả: 7


7

## Độ phức tạp của Cách 4

- Xây `adj` và DFS bằng stack: `O(n)`.
- Mỗi lần gộp một con vào cha chỉ làm vài phép tính trên mảng 2 phần tử: `O(1)`. Có `n - 1` lần gộp.

→ **Thời gian `O(n)`, bộ nhớ `O(n)`.** Cùng bậc với Cách 3, nhưng hằng số lớn hơn (phải xây cây, duyệt DFS) — đổi lại, nó **không cần** giả thiết chi phí dương.

# 11. Phiên bản nộp LeetCode

Ta nộp **Cách 3** vì nó ngắn nhất, nhanh nhất và đạt `O(n)`:

- Không cần danh sách kề, không cần DFS, không lo đệ quy.
- `v.bit_count()` cần Python ≥ 3.10 (LeetCode đáp ứng). Nếu môi trường cũ hơn, thay bằng `bin(v).count("1")`.
- Nhớ đổi tên method nếu skeleton trên LeetCode khác `minCost`.

In [26]:
from typing import List

class Solution:
    def minCost(self, edges: List[List[int]], values: List[int]) -> int:
        n = len(values)
        p = [v.bit_count() & 1 for v in values]
        INF = float("inf")
        best1 = [INF] * n
        best2 = [INF] * n
        ans = INF
        for u, v, c in edges:
            if p[u] == p[v]:
                if c < ans:
                    ans = c
            else:
                m = u if p[u] == 0 else v
                if c < best1[m]:
                    best2[m] = best1[m]
                    best1[m] = c
                elif c < best2[m]:
                    best2[m] = c
        for m in range(n):
            if best2[m] < INF:
                s = best1[m] + best2[m]
                if s < ans:
                    ans = s
        return -1 if ans == INF else ans

In [27]:
sol = Solution()

# 1) Ví dụ của đề
for edges, values, expected in EXAMPLES:
    assert sol.minCost(edges, values) == expected
print("3 ví dụ của đề: PASS")

# 2) Edge cases
edge_cases = [
    ("n=2, cùng parity",                [[0, 1, 7]], [3, 5], 7),          # 3,5 đều có 2 bit
    ("n=2, khác parity",                [[0, 1, 7]], [1, 3], -1),
    ("sao: tâm lẻ, lá chẵn",            [[0, i, i] for i in range(1, 6)], [1, 3, 5, 6, 9, 10], -1),
    ("sao: tâm chẵn, lá lẻ",            [[0, i, i] for i in range(1, 6)], [3, 1, 2, 4, 8, 7], 1 + 2),
    ("chi phí 10^9, đường 3 đỉnh",      [[0, 1, 10**9], [1, 2, 10**9]], [1, 3, 2], 2 * 10**9),
    ("giá trị lớn tới 10^9",            [[0, 1, 4], [1, 2, 4]], [10**9, 2**29, 2**30 - 1], 4),
]
for name, edges, values, expected in edge_cases:
    got = sol.minCost(edges, values)
    brute = min_cost_bruteforce(edges, values)
    assert got == expected == brute, (name, got, expected, brute)
    print(f"{name:<30} -> {got}  PASS")

3 ví dụ của đề: PASS
n=2, cùng parity               -> 7  PASS
n=2, khác parity               -> -1  PASS
sao: tâm lẻ, lá chẵn           -> -1  PASS
sao: tâm chẵn, lá lẻ           -> 3  PASS
chi phí 10^9, đường 3 đỉnh     -> 2000000000  PASS
giá trị lớn tới 10^9           -> 4  PASS


Giải thích vài edge case:

- `values = [1, 3]`: `1` có 1 bit (lẻ), `3` có 2 bit (chẵn) → đường duy nhất `0-1` có bits `1-0` → `-1`.
- Sao "tâm lẻ, lá chẵn": mọi đường là `lá - tâm` (bits `0-1`) hoặc `lá - tâm - lá` (bits `0-1-0`), đều lẻ → `-1`. Ta sẽ chứng minh ở mục 13 rằng đây là **hình dạng duy nhất** cho đáp án `-1`.
- Ca cuối: các giá trị lớn. Đừng đoán số bit 1 bằng mắt — hãy **in ra** (cell ngay dưới). `10^9` có 13 bit 1 (lẻ), `2^29` có 1 bit (lẻ), `2^30 - 1` có 30 bit (chẵn) → `p = [1, 1, 0]`. Cạnh `0-1` có bits `1-1` → hợp lệ, chi phí `4`. Cạnh `1-2` có bits `1-0` → không hợp lệ; đường `0-1-2` có bits `1-1-0` → hợp lệ nhưng đắt hơn (8). Đáp án `4`.

Sau cell in bit là bài test với `n = 10^5` trên ba hình dạng cây: ngẫu nhiên, hình sao (bậc rất lớn) và đường thẳng (rất sâu).

In [28]:
for x in [10**9, 2**29, 2**30 - 1]:
    print(f"{x:>13,}: bin = {bin(x)[2:]:>30}  bit_count = {x.bit_count():>2} -> parity {x.bit_count() & 1}")

1,000,000,000: bin = 111011100110101100101000000000  bit_count = 13 -> parity 1
  536,870,912: bin = 100000000000000000000000000000  bit_count =  1 -> parity 1
1,073,741,823: bin = 111111111111111111111111111111  bit_count = 30 -> parity 0


In [29]:
# Test với n = 10^5 trên 3 hình dạng cây (ngẫu nhiên, hình sao, đường thẳng):
# kiểm tra lời giải chạy nhanh, không lỗi đệ quy, và Cách 3 khớp Cách 4.
import time

def big_tree(kind, n, seed):
    rng = random.Random(seed)
    if kind == "random":
        edges = [[rng.randrange(i), i, rng.randint(1, 10**9)] for i in range(1, n)]
    elif kind == "star":
        edges = [[0, i, rng.randint(1, 10**9)] for i in range(1, n)]
    else:  # "line"
        edges = [[i, i + 1, rng.randint(1, 10**9)] for i in range(n - 1)]
    values = [rng.randint(1, 10**9) for _ in range(n)]
    return edges, values

for kind in ["random", "star", "line"]:
    edges, values = big_tree(kind, 10**5, seed=2026)
    t0 = time.perf_counter()
    r3 = sol.minCost(edges, values)
    t3 = time.perf_counter() - t0
    t0 = time.perf_counter()
    r4 = min_cost_tree_dp(edges, values)
    t4 = time.perf_counter() - t0
    assert r3 == r4
    print(f"{kind:<6}: đáp án = {r3:<12} Cách 3: {t3:.3f}s   Cách 4: {t4:.3f}s")

random: đáp án = 4381         Cách 3: 0.022s   Cách 4: 0.344s


star  : đáp án = 9943         Cách 3: 0.013s   Cách 4: 0.170s


line  : đáp án = 6338         Cách 3: 0.013s   Cách 4: 0.158s


# 12. Kiểm tra chéo (cross-check) với brute force

Ví dụ của đề chỉ có 3 ca — quá ít cho một bài Hard. Ta sinh **nhiều cây ngẫu nhiên nhỏ** (seed cố định để chạy lại ra cùng kết quả) và so **mọi cách** với brute force.

Ta trộn nhiều kiểu: cây ngẫu nhiên, cây hình sao, giá trị nhỏ (để có nhiều đỉnh trùng parity), chi phí nhỏ (để có nhiều đường bằng chi phí nhau).

In [30]:
def random_case(rng):
    n = rng.randint(2, 9)
    star = rng.random() < 0.2
    max_cost = rng.choice([1, 3, 10, 10**9])
    edges = []
    for i in range(1, n):
        par = 0 if star else rng.randrange(i)
        edges.append([par, i, rng.randint(1, max_cost)])
    # xáo trộn nhãn đỉnh và thứ tự cạnh để không phụ thuộc "đỉnh 0 là gốc"
    perm = list(range(n))
    rng.shuffle(perm)
    edges = [[perm[u], perm[v], c] for u, v, c in edges]
    rng.shuffle(edges)
    values = [rng.randint(1, rng.choice([3, 7, 15, 10**9])) for _ in range(n)]
    return edges, values

rng = random.Random(4100)
approaches = [min_cost_short_naive, min_cost_local, min_cost_tree_dp, Solution().minCost]
num_cases, num_minus1 = 600, 0
for _ in range(num_cases):
    edges, values = random_case(rng)
    expected = min_cost_bruteforce(edges, values)
    num_minus1 += expected == -1
    for f in approaches:
        got = f(edges, values)
        assert got == expected, (f.__name__, edges, values, got, expected)
print(f"{num_cases} ca ngẫu nhiên: cả 4 cách đều khớp brute force (trong đó {num_minus1} ca có đáp án -1)")

600 ca ngẫu nhiên: cả 4 cách đều khớp brute force (trong đó 53 ca có đáp án -1)


# 13. Vì sao các cách giải đúng?

## Bổ đề 1 (parity của XOR)

`popcount(a ^ b) = popcount(a) + popcount(b) − 2·popcount(a & b)`, nên `parity(a ^ b) = parity(a) ^ parity(b)`. Áp dụng liên tiếp: một đường hợp lệ ⟺ XOR các bit `p` trên đường bằng `0` ⟺ **số đỉnh lẻ trên đường là chẵn**. (Đã kiểm chứng bằng code ở mục 5.)

## Bổ đề 2 (đường rẻ nhất có 2 hoặc 3 đỉnh)

Mọi đường hợp lệ chứa một đoạn con hợp lệ là **cạnh cùng parity** hoặc **đường `lẻ - chẵn - lẻ`** (chứng minh ở mục 6: hoặc có hai đỉnh kề cùng bit, hoặc dãy bit luân phiên và khi đó có cửa sổ `1,0,1`). Đoạn con của một đường trên cây vẫn là một đường hợp lệ giữa hai đỉnh phân biệt, và vì mọi `c ≥ 1`, nó **không đắt hơn**. Vậy giá trị tối ưu luôn đạt được ở một đường 2 hoặc 3 đỉnh.

## Cách 3 đúng vì...

- Mọi cạnh cùng parity đều được xét → tính đúng `A`.
- Mọi đường `lẻ - m - lẻ` có hai cạnh đi ra từ đỉnh chẵn `m` tới hàng xóm lẻ. Mỗi cạnh khác parity được đưa vào **đúng đỉnh chẵn** của nó. Với `m` cố định, tổng nhỏ nhất của hai cạnh **khác nhau** là `best1[m] + best2[m]` → tính đúng `B`.
- Không có đường nào không hợp lệ bị tính: cạnh cùng parity luôn hợp lệ; `best1[m] + best2[m]` ứng với hai cạnh khác nhau tới hai hàng xóm lẻ khác nhau → đường 3 đỉnh `1-0-1` hợp lệ.

## Cách 4 đúng vì... (invariant)

> Sau khi đã gộp các con `v1, ..., vj` vào `u`: `down[u][q]` là chi phí nhỏ nhất của nhánh xuống từ `u` có parity `q`, chỉ dùng `u` và cây con của `v1..vj`; còn `ans` đã xét mọi đường có đỉnh cao nhất là `u` và nằm trong `u` + cây con `v1..vj`.

Khi gộp thêm `v(j+1)`, mỗi đường mới có đỉnh cao nhất `u` phải đi xuống `v(j+1)` ở một phía và phía kia nằm trong phần cũ (hoặc chỉ là `u`) — đó đúng là cặp `(down[u][a], cand[b])`. Mỗi đường được xét **đúng một lần** tại đỉnh cao nhất của nó.

## Hệ quả: khi nào đáp án là `-1`?

Đáp án `-1` ⟺ không có cạnh cùng parity **và** không đỉnh chẵn nào có ≥ 2 hàng xóm lẻ. Khi đó mọi đỉnh chẵn là **lá** (bậc 1), và không thể có hai đỉnh lẻ (đường nối chúng phải đi qua một đỉnh chẵn ở giữa có bậc ≥ 2). Vậy:

> Đáp án `-1` ⟺ cây là **hình sao** với **tâm là đỉnh lẻ** và mọi lá là đỉnh chẵn (gồm cả trường hợp `n = 2` với hai đỉnh khác parity).

Ví dụ 3 chính là trường hợp này. Cell dưới kiểm chứng trên cây ngẫu nhiên.

In [31]:
def is_odd_center_star(edges, values):
    p = [v.bit_count() & 1 for v in values]
    odd = [i for i in range(len(values)) if p[i] == 1]
    if len(odd) != 1:
        return False
    center = odd[0]
    return all(u == center or v == center for u, v, _ in edges)

rng = random.Random(7)
count_minus1 = 0
for _ in range(1500):
    n = rng.randint(2, 7)
    star = rng.random() < 0.5
    edges = [[0 if star else rng.randrange(i), i, rng.randint(1, 5)] for i in range(1, n)]
    values = [rng.randint(1, 7) for _ in range(n)]
    ans = min_cost_bruteforce(edges, values)
    assert (ans == -1) == is_odd_center_star(edges, values)
    count_minus1 += ans == -1
print("Đặc trưng '-1 ⟺ sao tâm lẻ, lá chẵn' đúng trên 1500 cây; số ca -1:", count_minus1)

Đặc trưng '-1 ⟺ sao tâm lẻ, lá chẵn' đúng trên 1500 cây; số ca -1: 153


# 14. Độ phức tạp và so sánh các cách

Ký hiệu: `n` = số đỉnh (số cạnh là `n - 1`), `deg(m)` = bậc của đỉnh `m`. Việc đếm bit của một số ≤ `10^9` (≤ 30 bit) coi như `O(1)`.

| Cách | Ý tưởng chính | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | DFS từ mọi đỉnh, XOR thật | `O(n²)` | `O(n)` | Hiển nhiên đúng; dùng làm trọng tài |
| 2. Đường 2–3 đỉnh ngây thơ | Mọi cạnh + mọi cặp hàng xóm | `O(n + Σ deg²)`, tệ nhất `O(n²)` | `O(n)` | TLE với cây hình sao |
| 3. Đường 2–3 đỉnh + top-2 | Một vòng qua cạnh, một vòng qua đỉnh | `O(n)` | `O(n)` | **Nộp bài**; cần chi phí ≥ 0 |
| 4. Tree DP chẵn/lẻ | Ghép 2 nhánh tại đỉnh cao nhất | `O(n)` | `O(n)` | Tổng quát; dùng cho biến thể |

Về tốc độ thực tế trong Python (xem cell test `n = 10^5` ở mục 11): cả Cách 3 và Cách 4 đều chạy dưới 1 giây với `n = 10^5`; Cách 3 nhanh hơn rõ rệt (thường gấp nhiều lần) vì không cần xây danh sách kề và không cần DFS. Với LeetCode, Cách 3 thoải mái trong giới hạn thời gian.

**Bộ nhớ:** Cách 3 dùng 3 mảng độ dài `n` (`p`, `best1`, `best2`). Cách 4 dùng `adj` (`2(n-1)` phần tử), `parent`, `order`, `down`.

# 15. Những lỗi thường gặp

## Lỗi 1 — Dùng tổng `+` thay cho XOR `^`

Đề nói "bitwise XOR of the values", không phải tổng. Ví dụ đường `1 - 0 - 3` của Ví dụ 1: XOR `1 ^ 3 ^ 1 = 3` (2 bit, hợp lệ), còn tổng `1 + 3 + 1 = 5` (cũng 2 bit — trùng hợp may mắn!). Với các đường khác, tổng và XOR cho kết quả khác hẳn, nên đáp án có thể sai.

## Lỗi 2 — Quên điều kiện "hai đỉnh phân biệt"

Nếu cho phép đường chỉ gồm 1 đỉnh `s` (chi phí 0), thì mỗi đỉnh có giá trị với số bit 1 chẵn (như `3`) sẽ cho đáp án `0` — sai.

## Lỗi 3 — Chỉ xét đường 2 đỉnh (cạnh)

Nghĩ rằng "đường ngắn nhất là một cạnh" → quên mẫu `lẻ - chẵn - lẻ`. Ví dụ 1 sẽ ra `-1` thay vì `7`.

Cell dưới chạy cả ba phiên bản sai trên các ví dụ của đề.

In [32]:
def wrong_sum_instead_of_xor(edges, values):
    n = len(values)
    adj = build_adj(n, edges)
    best = -1
    for s in range(n):
        stack = [(s, -1, values[s], 0)]
        while stack:
            u, par, total, cost = stack.pop()
            if u != s and bin(total).count("1") % 2 == 0:        # SAI: total là tổng
                best = cost if best == -1 else min(best, cost)
            for v, c in adj[u]:
                if v != par:
                    stack.append((v, u, total + values[v], cost + c))
    return best

def wrong_allow_single_node(edges, values):
    n = len(values)
    adj = build_adj(n, edges)
    best = -1
    for s in range(n):
        stack = [(s, -1, values[s], 0)]
        while stack:
            u, par, x, cost = stack.pop()
            if bin(x).count("1") % 2 == 0:                        # SAI: thiếu u != s
                best = cost if best == -1 else min(best, cost)
            for v, c in adj[u]:
                if v != par:
                    stack.append((v, u, x ^ values[v], cost + c))
    return best

def wrong_edges_only(edges, values):
    p = [v.bit_count() & 1 for v in values]
    costs = [c for u, v, c in edges if p[u] == p[v]]                # SAI: quên đường 3 đỉnh
    return min(costs) if costs else -1

for k, (edges, values, expected) in enumerate(EXAMPLES, start=1):
    print(f"Ví dụ {k}: đúng = {expected:>2} | tổng thay XOR = {wrong_sum_instead_of_xor(edges, values):>2}"
          f" | cho phép 1 đỉnh = {wrong_allow_single_node(edges, values):>2}"
          f" | chỉ xét cạnh = {wrong_edges_only(edges, values):>2}")

Ví dụ 1: đúng =  7 | tổng thay XOR =  6 | cho phép 1 đỉnh =  0 | chỉ xét cạnh = -1
Ví dụ 2: đúng =  5 | tổng thay XOR = 10 | cho phép 1 đỉnh =  0 | chỉ xét cạnh =  5
Ví dụ 3: đúng = -1 | tổng thay XOR =  7 | cho phép 1 đỉnh =  0 | chỉ xét cạnh = -1


Output cho thấy: "tổng thay XOR" sai ở **cả ba** ví dụ (6, 10, 7 thay vì 7, 5, -1), "cho phép 1 đỉnh" ra `0` ở cả ba ví dụ, "chỉ xét cạnh" sai ở Ví dụ 1 (ra `-1` thay vì `7`).

## Lỗi 4 — Lấy đỉnh giữa là đỉnh **lẻ**

Đường 3 đỉnh cần xét là `lẻ - chẵn - lẻ`. Nếu lấy nhầm đỉnh giữa lẻ với hai hàng xóm chẵn (`chẵn - lẻ - chẵn`, XOR = 1) thì đang cộng một đường **không hợp lệ**. Ví dụ 3 (tâm `0` lẻ) sẽ ra `5 + 7 = 12` thay vì `-1`.

(Thú vị: nếu đỉnh giữa **chẵn** mà ta quên lọc "hàng xóm phải lẻ", kết quả vẫn đúng! Vì nếu một trong hai cạnh rẻ nhất đi tới hàng xóm chẵn, thì riêng cạnh đó đã là đường 2 đỉnh hợp lệ còn rẻ hơn. Nhưng lọc rõ ràng thì dễ chứng minh hơn.)

## Lỗi 5 — Tree DP: cập nhật `down[u]` **trước** khi ghép

Nếu (c) chạy trước (b), nhánh qua `v` bị ghép với **chính nó** → "đường" `v - u - v` đi qua cạnh `u-v` hai lần. Ví dụ: `n = 2`, `values = [3, 1]` (`p = [0, 1]`), cạnh chi phí 5. Đáp án đúng `-1`, phiên bản sai ra `10`.

## Lỗi 6 — Tạo danh sách kề bằng `[[]] * n`

`[[]] * n` tạo `n` **tham chiếu tới cùng một list**! Thêm vào `adj[0]` cũng là thêm vào `adj[1]`, ... Luôn dùng `[[] for _ in range(n)]`.

In [33]:
def wrong_odd_middle(edges, values):
    n = len(values)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    best1, best2 = [INF] * n, [INF] * n
    ans = INF
    for u, v, c in edges:
        if p[u] == p[v]:
            ans = min(ans, c)
        else:
            for m in (u, v):                     # SAI: cho cả đầu lẻ làm đỉnh giữa
                if c < best1[m]:
                    best2[m], best1[m] = best1[m], c
                elif c < best2[m]:
                    best2[m] = c
    for m in range(n):
        if best2[m] < INF:
            ans = min(ans, best1[m] + best2[m])
    return -1 if ans == INF else ans

def wrong_tree_dp_merge_first(edges, values):
    n = len(values)
    adj = build_adj(n, edges)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    parent, parent_cost, order = root_tree(adj, 0)
    down = [[INF, INF] for _ in range(n)]
    for u in range(n):
        down[u][p[u]] = 0
    ans = INF
    for v in reversed(order):
        u = parent[v]
        if u == -1:
            continue
        cand = [INF, INF]
        for q in (0, 1):
            cand[q ^ p[u]] = min(cand[q ^ p[u]], down[v][q] + parent_cost[v])
        for q in (0, 1):                          # SAI: cập nhật trước ...
            down[u][q] = min(down[u][q], cand[q])
        for a in (0, 1):                          # ... rồi mới ghép
            ans = min(ans, down[u][a] + cand[a ^ p[u]])
    return -1 if ans == INF else ans

def local_without_neighbor_filter(edges, values):
    """Đỉnh giữa chẵn nhưng KHÔNG lọc hàng xóm lẻ: lấy 2 cạnh rẻ nhất bất kỳ."""
    n = len(values)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    best1, best2 = [INF] * n, [INF] * n
    ans = INF
    for u, v, c in edges:
        if p[u] == p[v]:
            ans = min(ans, c)
        for m in (u, v):
            if p[m] == 0:                        # chỉ lọc đỉnh giữa chẵn
                if c < best1[m]:
                    best2[m], best1[m] = best1[m], c
                elif c < best2[m]:
                    best2[m] = c
    for m in range(n):
        if best2[m] < INF:
            ans = min(ans, best1[m] + best2[m])
    return -1 if ans == INF else ans

e3, v3, exp3 = EXAMPLES[2]
print("Lỗi 4 trên Ví dụ 3: sai =", wrong_odd_middle(e3, v3), "| đúng =", exp3)
same = 0
for s in range(300):
    e, v = random_case(random.Random(s))
    same += local_without_neighbor_filter(e, v) == min_cost_bruteforce(e, v)
print("Không lọc hàng xóm lẻ (đỉnh giữa chẵn): đúng", same, "/ 300 ca -> vẫn đúng như đã giải thích")
print("Lỗi 5 trên n=2    : sai =", wrong_tree_dp_merge_first([[0, 1, 5]], [3, 1]),
      "| đúng =", min_cost_tree_dp([[0, 1, 5]], [3, 1]))

bad_adj = [[]] * 3
bad_adj[0].append((1, 5))
print("Lỗi 6: [[]] * 3 sau khi thêm vào adj[0]:", bad_adj)
good_adj = [[] for _ in range(3)]
good_adj[0].append((1, 5))
print("       [[] for _ in range(3)]          :", good_adj)

Lỗi 4 trên Ví dụ 3: sai = 12 | đúng = -1
Không lọc hàng xóm lẻ (đỉnh giữa chẵn): đúng 300 / 300 ca -> vẫn đúng như đã giải thích
Lỗi 5 trên n=2    : sai = 10 | đúng = -1
Lỗi 6: [[]] * 3 sau khi thêm vào adj[0]: [[(1, 5)], [(1, 5)], [(1, 5)]]
       [[] for _ in range(3)]          : [[(1, 5)], [], []]


## Các lỗi nhỏ khác cần nhớ

- **Quên đổi `INF` thành `-1`**: hàm trả `inf` (kiểu `float`) khi không có đường → sai định dạng. Luôn kết thúc bằng `return -1 if ans == INF else ans`.
- **Chỉ thêm cạnh một chiều** vào `adj` (`adj[u].append(v)` mà quên `adj[v].append(u)`): DFS sẽ không đi được hết cây.
- **Dùng đệ quy cho DFS** với `n = 10^5`: `RecursionError` trên cây đường thẳng (mục 3.6).
- **Viết `5.bit_count()`**: lỗi cú pháp; phải là `(5).bit_count()`.
- **C++/Java: tràn số** khi cộng hai chi phí `10^9` (tổng `2·10^9 > 2^31 − 1`) — dùng `long long`/`long`. Python không gặp vấn đề này.
- **Dùng Bổ đề 2 khi chi phí có thể âm**: bổ đề chỉ đúng khi mọi chi phí ≥ 0. Đề này có `c ≥ 1` nên an toàn; với biến thể khác hãy dùng Tree DP.

# 16. Dấu hiệu nhận dạng và bài liên quan

## Dấu hiệu 1 — "số bit 1 chẵn/lẻ" của một XOR (hoặc tổng)

Ngay khi đề chỉ hỏi **chẵn/lẻ** của popcount, hãy thay mỗi số bằng **một bit** `popcount & 1`: popcount-parity "đi xuyên qua" XOR. Tương tự, nếu đề hỏi **tổng** chẵn/lẻ, thay mỗi số bằng `x & 1`.

## Dấu hiệu 2 — "chi phí nhỏ nhất", chi phí dương, điều kiện kiểm tra được trên đoạn ngắn

Hãy tự hỏi: *"Đáp án tối ưu có thể dài không? Nếu dài, nó có chứa một đoạn con hợp lệ ngắn hơn không?"* Nếu có, đáp án tối ưu là một **cấu trúc nhỏ** (cạnh, bộ 3 đỉnh, ...) → chỉ cần duyệt cục bộ.

## Dấu hiệu 3 — "đường đi giữa hai đỉnh bất kỳ trên cây" với điều kiện dạng XOR/tổng modulo nhỏ

Dùng **Tree DP**: mỗi đường được đếm tại **đỉnh cao nhất**, ghép **hai nhánh đi xuống**, trạng thái là giá trị XOR/modulo (ở đây chỉ 2 trạng thái). Hoặc dùng **XOR tiền tố từ gốc** (`xor_root[u]`) + LCA.

## Bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 191 — Number of 1 Bits | Đếm bit 1 (popcount) |
| LeetCode 461 — Hamming Distance | `popcount(x ^ y)` |
| LeetCode 1310 — XOR Queries of a Subarray | XOR tiền tố, tính chất `a ^ a = 0` |
| LeetCode 124 — Binary Tree Maximum Path Sum | Tree DP: ghép hai nhánh tại đỉnh cao nhất |
| LeetCode 2246 — Longest Path With Different Adjacent Characters | Tree DP + giữ 2 giá trị tốt nhất từ các con |
| LeetCode 2791 — Count Paths That Can Form a Palindrome in a Tree | Đường trên cây + XOR bitmask từ gốc |

## Biến thể mini: đường có số bit 1 **lẻ** rẻ nhất

Lập luận y hệt mục 6. Một đường có parity 1 thì **hoặc** có **hai đỉnh kề khác bit** (một cạnh khác parity, XOR = 1 — đường 2 đỉnh hợp lệ), **hoặc** mọi bit trên đường đều giống nhau. Trường hợp sau: toàn bit `0` thì XOR = 0 (loại), nên phải toàn bit `1`, và XOR = 1 nghĩa là **số lượng đỉnh là số lẻ**, tức ít nhất 3 → chứa cửa sổ `1-1-1` (đường 3 đỉnh hợp lệ). Vậy:

```text
đáp án_lẻ = min( cạnh khác parity rẻ nhất,
                 min trên đỉnh LẺ m của tổng 2 cạnh rẻ nhất tới hàng xóm LẺ )
```

In [34]:
def min_cost_odd_local(edges, values):
    """Biến thể: đường có XOR với số bit 1 LẺ, chi phí nhỏ nhất."""
    n = len(values)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    best1, best2 = [INF] * n, [INF] * n
    ans = INF
    for u, v, c in edges:
        if p[u] != p[v]:
            ans = min(ans, c)                     # cạnh khác parity: XOR = 1
        elif p[u] == 1:                           # cạnh lẻ - lẻ: góp vào bộ 1-1-1
            for m in (u, v):
                if c < best1[m]:
                    best2[m], best1[m] = best1[m], c
                elif c < best2[m]:
                    best2[m] = c
    for m in range(n):
        if p[m] == 1 and best2[m] < INF:
            ans = min(ans, best1[m] + best2[m])
    return -1 if ans == INF else ans

def min_cost_odd_bruteforce(edges, values):
    n = len(values)
    adj = build_adj(n, edges)
    best = -1
    for s in range(n):
        stack = [(s, -1, values[s], 0)]
        while stack:
            u, par, x, cost = stack.pop()
            if u != s and x.bit_count() % 2 == 1:
                best = cost if best == -1 else min(best, cost)
            for v, c in adj[u]:
                if v != par:
                    stack.append((v, u, x ^ values[v], cost + c))
    return best

rng = random.Random(11)
for _ in range(400):
    e, v = random_case(rng)
    assert min_cost_odd_local(e, v) == min_cost_odd_bruteforce(e, v)
print("Biến thể 'lẻ' khớp brute force trên 400 ca")
print("Ví dụ 1, đường lẻ rẻ nhất:", min_cost_odd_local(EXAMPLES[0][0], EXAMPLES[0][1]))

Biến thể 'lẻ' khớp brute force trên 400 ca
Ví dụ 1, đường lẻ rẻ nhất: 3


# 17. Bài tập

Làm theo thứ tự, từ dễ đến khó. Các hàm TODO **không được gọi** khi chạy notebook; test đã được comment sẵn — làm xong thì bỏ comment để tự kiểm tra. Đáp án nằm ở mục **Đáp án tham khảo** cuối notebook.

## Bài 1 — Đoán output (không chạy code)

Hãy đoán kết quả của từng biểu thức:

```python
5 ^ 3
bin(6)
(6).bit_count()
(13).bit_count() & 1
(1 ^ 3 ^ 7).bit_count() % 2
[v.bit_count() & 1 for v in [8, 15, 6, 1]]
```

## Bài 2 — Tính tay

Cho cây:

```text
edges  = [[0,1,3],[1,2,2],[2,3,4],[1,4,7],[4,5,10]]
values = [2, 6, 1, 3, 4, 7]
```

1. Tính mảng `p` (parity số bit 1 của từng giá trị).
2. Liệt kê các cạnh cùng parity → tính `A`.
3. Với mỗi đỉnh chẵn, tìm 2 cạnh rẻ nhất tới hàng xóm lẻ → tính `B`.
4. Đáp án là bao nhiêu? Đường đi nào đạt đáp án?

### Gợi ý Bài 1–2

- Mức 1 (hướng tư duy): viết từng số ra nhị phân trước; với Bài 2, đi theo đúng công thức `min(A, B)` ở mục 6.
- Mức 2 (cấu trúc): `5 = 101`, `3 = 011`; `bin` trả về **chuỗi** có tiền tố `"0b"`; `13 = 1101`.
- Mức 3 (gần đáp án): Bài 2 có đúng một cạnh cùng parity (giữa hai đỉnh lẻ); đỉnh chẵn `1` có 3 hàng xóm lẻ.

## Bài 3 — Điền TODO: viết lại Cách 3

Hoàn thiện hàm dưới đây (không nhìn lại mục 8 nếu có thể).

### Gợi ý Bài 3

- Mức 1: mỗi cạnh chỉ có 2 khả năng: cùng parity (đường 2 đỉnh) hoặc khác parity (góp vào đỉnh chẵn).
- Mức 2: cần 3 mảng/biến: `p`, `best1`, `best2` và `ans = INF`; cập nhật top-2 theo mục 3.7.
- Mức 3: `m = u if p[u] == 0 else v`; cuối cùng `for m in range(n): if best2[m] < INF: ans = min(ans, best1[m] + best2[m])`.

In [35]:
def exercise_3_min_cost(edges: List[List[int]], values: List[int]) -> int:
    # TODO 1: tính p[i] = values[i].bit_count() & 1
    # TODO 2: khởi tạo INF, best1, best2, ans
    # TODO 3: duyệt từng cạnh (u, v, c):
    #         - nếu p[u] == p[v]: cập nhật ans
    #         - ngược lại: m = đầu chẵn, cập nhật best1[m], best2[m]
    # TODO 4: duyệt từng đỉnh m, nếu best2[m] < INF thì cập nhật ans
    # TODO 5: trả về -1 nếu ans vẫn là INF
    pass

# Làm xong rồi bỏ comment để kiểm tra:
# for edges, values, expected in EXAMPLES:
#     assert exercise_3_min_cost(edges, values) == expected
# print("Bài 3 PASS")

## Bài 4 — Biến thể: trả về chính đường đi tối ưu

Viết hàm trả về **danh sách đỉnh** của một đường Balanced-XOR có chi phí nhỏ nhất (theo thứ tự đi trên đường), hoặc `[]` nếu không có. Nếu có nhiều đường tối ưu, trả về đường nào cũng được.

Ví dụ: Ví dụ 1 → `[1, 0, 3]` hoặc `[3, 0, 1]`; Ví dụ 2 → `[0, 1]` hoặc `[1, 0]`; Ví dụ 3 → `[]`.

### Gợi ý Bài 4

- Mức 1: đáp án chỉ có 2 hoặc 3 đỉnh — chỉ cần nhớ thêm "ai" tạo ra giá trị nhỏ nhất.
- Mức 2: song song với `best1[m]`, `best2[m]`, lưu thêm `who1[m]`, `who2[m]` = hàng xóm lẻ tương ứng; khi cập nhật `ans` từ cạnh, lưu `path = [u, v]`.
- Mức 3: khi cập nhật từ đỉnh giữa: `path = [who1[m], m, who2[m]]`. Nhớ "đẩy" `who1` xuống `who2` cùng lúc với `best1` xuống `best2`.

In [36]:
def exercise_4_best_path(edges: List[List[int]], values: List[int]) -> List[int]:
    # TODO: như Cách 3 nhưng lưu thêm hàng xóm tạo ra best1/best2
    #       và lưu lại đường đi mỗi khi ans được cập nhật
    raise NotImplementedError("Hãy tự hoàn thiện Bài 4")

# Kiểm tra (bỏ comment sau khi làm):
# print(exercise_4_best_path(*EXAMPLES[0][:2]))   # [1, 0, 3] hoặc [3, 0, 1]
# print(exercise_4_best_path(*EXAMPLES[1][:2]))   # [0, 1] hoặc [1, 0]
# print(exercise_4_best_path(*EXAMPLES[2][:2]))   # []

## Bài 5 — Biến thể: đếm số đường Balanced-XOR

Đếm có bao nhiêu **cặp không thứ tự** `{u, v}` (u ≠ v) mà đường giữa chúng là Balanced-XOR.

Ví dụ 1 → `5`, Ví dụ 2 → `2`, Ví dụ 3 → `0`.

Lưu ý: bài **đếm** thì Bổ đề 2 **không** giúp được (đường dài cũng phải đếm). Bản `O(n²)` (DFS từ mọi đỉnh) là bước đầu tốt; thử thách là `O(n)` bằng Tree DP.

### Gợi ý Bài 5

- Mức 1: giống Cách 4, nhưng thay "chi phí nhỏ nhất" bằng "**số lượng** nhánh" có parity `q`.
- Mức 2: `cnt[u][q]` = số nhánh xuống từ `u` (kể cả nhánh chỉ gồm `u`) có parity `q`. Khởi tạo `cnt[u][p[u]] = 1`. Ghép: dùng **nhân** thay cho cộng, **cộng dồn** thay cho `min`.
- Mức 3: khi gộp `v` vào `u`: `cand[q ^ p[u]] += cnt[v][q]`; `total += cnt[u][a] * cand[a ^ p[u]]` (với `a = 0, 1`), **rồi mới** `cnt[u][q] += cand[q]`.

In [37]:
def exercise_5_count_balanced(edges: List[List[int]], values: List[int]) -> int:
    # TODO: Tree DP đếm, O(n)
    #   - p, adj, root_tree(adj, 0)
    #   - cnt = [[0, 0] for _ in range(n)]; cnt[u][p[u]] = 1
    #   - duyệt reversed(order), gộp con vào cha: ghép trước, cộng dồn sau
    pass

# Kiểm tra (bỏ comment sau khi làm):
# assert [exercise_5_count_balanced(e, v) for e, v, _ in EXAMPLES] == [5, 2, 0]

## Bài 6 — Mini challenge: chi phí **lớn nhất** của một đường Balanced-XOR

Giữ nguyên đề nhưng hỏi chi phí **lớn nhất** (không có thì `-1`).

Ví dụ 1 → `14` (đường `2 - 0 - 3 - 4`), Ví dụ 2 → `15`, Ví dụ 3 → `-1`.

Câu hỏi tư duy trước khi code: vì sao Cách 3 (đường 2–3 đỉnh) **không dùng được** ở đây?

### Gợi ý Bài 6

- Mức 1: chi phí dương nên đường **dài hơn** thường đắt hơn → đường tối ưu có thể rất dài → cần Tree DP.
- Mức 2: đổi `min` thành `max`, `INF` thành `NEG = float("-inf")` (nghĩa là "chưa có nhánh với parity này").
- Mức 3: `down[u][p[u]] = 0`; `cand[q ^ p[u]] = max(cand[q ^ p[u]], down[v][q] + c)`; `ans = max(ans, down[u][a] + cand[a ^ p[u]])`; cuối cùng `return -1 if ans == NEG else ans`.

In [38]:
def exercise_6_max_cost(edges: List[List[int]], values: List[int]) -> int:
    # TODO: Tree DP với max thay cho min
    raise NotImplementedError("Hãy tự hoàn thiện Bài 6")

# Kiểm tra (bỏ comment sau khi làm):
# assert [exercise_6_max_cost(e, v) for e, v, _ in EXAMPLES] == [14, 15, -1]

# Đáp án tham khảo

Hãy tự làm trước khi đọc phần này.

## Đáp án Bài 1

| Biểu thức | Kết quả | Giải thích |
|---|---|---|
| `5 ^ 3` | `6` | `101 ^ 011 = 110` |
| `bin(6)` | `'0b110'` | chuỗi, có tiền tố `0b` |
| `(6).bit_count()` | `2` | `110` có 2 bit 1 |
| `(13).bit_count() & 1` | `1` | `1101` có 3 bit 1 → lẻ |
| `(1 ^ 3 ^ 7).bit_count() % 2` | `0` | `1 ^ 3 ^ 7 = 5 = 101` → 2 bit |
| `[v.bit_count() & 1 for v in [8, 15, 6, 1]]` | `[1, 0, 0, 1]` | 8 có 1 bit, 15 có 4, 6 có 2, 1 có 1 |

In [39]:
print(5 ^ 3)
print(bin(6))
print((6).bit_count())
print((13).bit_count() & 1)
print((1 ^ 3 ^ 7).bit_count() % 2)
print([v.bit_count() & 1 for v in [8, 15, 6, 1]])

6
0b110
2
1
0
[1, 0, 0, 1]


## Đáp án Bài 2

1. `values = [2, 6, 1, 3, 4, 7]` → số bit 1 `= [1, 2, 1, 2, 1, 3]` → `p = [1, 0, 1, 0, 1, 1]`.
2. Cạnh `0-1` (1,0), `1-2` (0,1), `2-3` (1,0), `1-4` (0,1) đều khác parity. Chỉ cạnh `4-5` (1,1) cùng parity → `A = 10`.
3. Đỉnh chẵn `1`: hàng xóm lẻ `0` (3), `2` (2), `4` (7) → hai cạnh rẻ nhất `2 + 3 = 5`. Đỉnh chẵn `3`: chỉ có hàng xóm `2` → bỏ qua. → `B = 5`.
4. Đáp án `min(10, 5) = 5`, đường `0 - 1 - 2` (hoặc `2 - 1 - 0`).

In [40]:
ex2_edges = [[0, 1, 3], [1, 2, 2], [2, 3, 4], [1, 4, 7], [4, 5, 10]]
ex2_values = [2, 6, 1, 3, 4, 7]
print("p =", [v.bit_count() & 1 for v in ex2_values])
assert min_cost_local(ex2_edges, ex2_values) == 5 == min_cost_bruteforce(ex2_edges, ex2_values)
print("Bài 2: đáp án 5 — PASS")

p = [1, 0, 1, 0, 1, 1]
Bài 2: đáp án 5 — PASS


## Đáp án Bài 3

In [41]:
def answer_3_min_cost(edges: List[List[int]], values: List[int]) -> int:
    n = len(values)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    best1, best2 = [INF] * n, [INF] * n
    ans = INF
    for u, v, c in edges:
        if p[u] == p[v]:
            ans = min(ans, c)
        else:
            m = u if p[u] == 0 else v
            if c < best1[m]:
                best2[m], best1[m] = best1[m], c
            elif c < best2[m]:
                best2[m] = c
    for m in range(n):
        if best2[m] < INF:
            ans = min(ans, best1[m] + best2[m])
    return -1 if ans == INF else ans

for edges, values, expected in EXAMPLES:
    assert answer_3_min_cost(edges, values) == expected
rng = random.Random(3)
for _ in range(300):
    e, v = random_case(rng)
    assert answer_3_min_cost(e, v) == min_cost_bruteforce(e, v)
print("Bài 3 PASS (3 ví dụ + 300 ca ngẫu nhiên)")

Bài 3 PASS (3 ví dụ + 300 ca ngẫu nhiên)


## Đáp án Bài 4

Để kiểm tra, ta viết hàm `check_path` xác nhận: các cặp đỉnh liên tiếp đều là cạnh, đường có ≥ 2 đỉnh phân biệt, XOR có số bit 1 chẵn, và chi phí bằng đáp án tối ưu của brute force.

In [42]:
def answer_4_best_path(edges: List[List[int]], values: List[int]) -> List[int]:
    n = len(values)
    p = [v.bit_count() & 1 for v in values]
    INF = float("inf")
    best1, best2 = [INF] * n, [INF] * n
    who1, who2 = [-1] * n, [-1] * n
    ans, path = INF, []
    for u, v, c in edges:
        if p[u] == p[v]:
            if c < ans:
                ans, path = c, [u, v]
        else:
            m, other = (u, v) if p[u] == 0 else (v, u)
            if c < best1[m]:
                best2[m], who2[m] = best1[m], who1[m]
                best1[m], who1[m] = c, other
            elif c < best2[m]:
                best2[m], who2[m] = c, other
    for m in range(n):
        if best2[m] < INF and best1[m] + best2[m] < ans:
            ans, path = best1[m] + best2[m], [who1[m], m, who2[m]]
    return path

def check_path(edges, values, path):
    expected = min_cost_bruteforce(edges, values)
    if expected == -1:
        return path == []
    cost_of = {}
    for u, v, c in edges:
        cost_of[(u, v)] = cost_of[(v, u)] = c
    if len(path) < 2 or len(set(path)) != len(path):
        return False
    x, cost = 0, 0
    for node in path:
        x ^= values[node]
    for a, b in zip(path, path[1:]):
        if (a, b) not in cost_of:
            return False
        cost += cost_of[(a, b)]
    return x.bit_count() % 2 == 0 and cost == expected

for edges, values, _ in EXAMPLES:
    print(answer_4_best_path(edges, values))
rng = random.Random(4)
for _ in range(300):
    e, v = random_case(rng)
    assert check_path(e, v, answer_4_best_path(e, v))
print("Bài 4 PASS (300 ca ngẫu nhiên)")

[3, 0, 1]
[0, 1]
[]
Bài 4 PASS (300 ca ngẫu nhiên)


## Đáp án Bài 5

Bản `O(n)` bằng Tree DP đếm, so với bản `O(n²)` đếm trực tiếp (DFS từ mọi đỉnh, mỗi cặp chỉ đếm khi `u > s`).

In [43]:
def answer_5_count_balanced(edges: List[List[int]], values: List[int]) -> int:
    n = len(values)
    adj = build_adj(n, edges)
    p = [v.bit_count() & 1 for v in values]
    parent, _, order = root_tree(adj, 0)
    cnt = [[0, 0] for _ in range(n)]
    for u in range(n):
        cnt[u][p[u]] = 1
    total = 0
    for v in reversed(order):
        u = parent[v]
        if u == -1:
            continue
        cand = [0, 0]
        for q in (0, 1):
            cand[q ^ p[u]] += cnt[v][q]
        for a in (0, 1):                         # ghép trước
            total += cnt[u][a] * cand[a ^ p[u]]
        for q in (0, 1):                         # cộng dồn sau
            cnt[u][q] += cand[q]
    return total

def count_balanced_bruteforce(edges, values):
    return len(all_balanced_paths(edges, values))   # hàm ở mục 6, mỗi cặp đếm 1 lần

print([answer_5_count_balanced(e, v) for e, v, _ in EXAMPLES])
assert [answer_5_count_balanced(e, v) for e, v, _ in EXAMPLES] == [5, 2, 0]
rng = random.Random(5)
for _ in range(300):
    e, v = random_case(rng)
    assert answer_5_count_balanced(e, v) == count_balanced_bruteforce(e, v)
print("Bài 5 PASS (300 ca ngẫu nhiên)")

[5, 2, 0]
Bài 5 PASS (300 ca ngẫu nhiên)


## Đáp án Bài 6

Cách 3 không dùng được vì Bổ đề 2 chỉ nói "đường **rẻ nhất** ngắn". Với câu hỏi **lớn nhất**, ta muốn đường càng dài càng tốt (chi phí dương), nên phải xét mọi đường → Tree DP với `max`.

In [44]:
def answer_6_max_cost(edges: List[List[int]], values: List[int]) -> int:
    n = len(values)
    adj = build_adj(n, edges)
    p = [v.bit_count() & 1 for v in values]
    NEG = float("-inf")
    parent, parent_cost, order = root_tree(adj, 0)
    down = [[NEG, NEG] for _ in range(n)]
    for u in range(n):
        down[u][p[u]] = 0
    ans = NEG
    for v in reversed(order):
        u = parent[v]
        if u == -1:
            continue
        c = parent_cost[v]
        cand = [NEG, NEG]
        for q in (0, 1):
            cand[q ^ p[u]] = max(cand[q ^ p[u]], down[v][q] + c)
        for a in (0, 1):
            ans = max(ans, down[u][a] + cand[a ^ p[u]])
        for q in (0, 1):
            down[u][q] = max(down[u][q], cand[q])
    return -1 if ans == NEG else ans

def max_cost_bruteforce(edges, values):
    paths = all_balanced_paths(edges, values)
    return max(cost for cost, _ in paths) if paths else -1

print([answer_6_max_cost(e, v) for e, v, _ in EXAMPLES])
assert [answer_6_max_cost(e, v) for e, v, _ in EXAMPLES] == [14, 15, -1]
rng = random.Random(6)
for _ in range(300):
    e, v = random_case(rng)
    assert answer_6_max_cost(e, v) == max_cost_bruteforce(e, v)
print("Bài 6 PASS (300 ca ngẫu nhiên)")

[14, 15, -1]
Bài 6 PASS (300 ca ngẫu nhiên)


# Tóm tắt

## Hai câu thần chú của bài này

1. **"Chẵn/lẻ của số bit 1 đi xuyên qua XOR."** → Thay mỗi giá trị bằng một bit `p = popcount & 1`. Đường hợp lệ ⟺ số đỉnh lẻ trên đường là chẵn.
2. **"Chi phí dương ⟹ đường rẻ nhất rất ngắn."** → Chỉ cần xét **cạnh cùng parity** và **bộ ba `lẻ - chẵn - lẻ`**.

## Template nhớ nhanh (Cách 3)

```python
p = [v.bit_count() & 1 for v in values]
INF = float("inf"); best1 = [INF] * n; best2 = [INF] * n; ans = INF
for u, v, c in edges:
    if p[u] == p[v]:
        ans = min(ans, c)                       # đường 2 đỉnh
    else:
        m = u if p[u] == 0 else v               # đỉnh giữa chẵn
        if c < best1[m]:   best2[m], best1[m] = best1[m], c
        elif c < best2[m]: best2[m] = c
for m in range(n):
    if best2[m] < INF:
        ans = min(ans, best1[m] + best2[m])     # đường lẻ - chẵn - lẻ
return -1 if ans == INF else ans
```

## Template Tree DP (khi biến thể không cho phép "đường ngắn")

```python
down[u][p[u]] = 0                     # nhánh chỉ gồm u
for v in reversed(order):             # con trước, cha sau
    u, c = parent[v], parent_cost[v]
    cand[q ^ p[u]] = down[v][q] + c   # nhánh mới qua v
    ans = best(ans, down[u][a] + cand[a ^ p[u]])   # GHÉP TRƯỚC
    down[u][q] = best(down[u][q], cand[q])         # CẬP NHẬT SAU
```

# Checklist trước khi submit

- [ ] Đã dùng XOR `^` (không phải tổng `+`) và chỉ quan tâm **chẵn/lẻ** của số bit 1?
- [ ] Đã loại đường chỉ có **một đỉnh** (đề yêu cầu hai đỉnh phân biệt)?
- [ ] Đã xét **cả** đường 2 đỉnh (cạnh cùng parity) **và** đường 3 đỉnh `lẻ - chẵn - lẻ`?
- [ ] Đỉnh giữa của đường 3 đỉnh là đỉnh **chẵn**, và dùng **hai cạnh khác nhau**?
- [ ] Không duyệt mọi cặp hàng xóm (cây hình sao sẽ TLE)?
- [ ] Trả về `-1` (không phải `inf`) khi không có đường nào?
- [ ] Nếu dùng DFS: không đệ quy, danh sách kề tạo bằng `[[] for _ in range(n)]`, thêm cạnh **hai chiều**?
- [ ] Nếu dùng Tree DP: **ghép trước, cập nhật sau**?
- [ ] Tên method và thứ tự tham số khớp skeleton trên LeetCode?
- [ ] Đã chạy 3 ví dụ của đề + edge case `n = 2` + cây hình sao tâm lẻ?